In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
from networkx.algorithms import bipartite
from collections import Counter


color_map_cat = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black', 'ROE': 'orange'}
color_map_cat_no_roe = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black'}

In [ ]:
# nuovo compare_layers
import math
import os
import random
import warnings
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter

try:
    import igraph as ig
    _HAS_IGRAPH = True
except Exception:
    ig = None
    _HAS_IGRAPH = False
try:
    import leidenalg as la
    _HAS_LEIDENALG = True
except Exception:
    la = None
    _HAS_LEIDENALG = False

def _js_from_counts(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    L = max(len(a), len(b))
    if len(a) < L: a = np.pad(a, (0, L-len(a)))
    if len(b) < L: b = np.pad(b, (0, L-len(b)))
    a = a + 1e-12; b = b + 1e-12
    a = a / a.sum(); b = b / b.sum()
    m = 0.5*(a+b)
    def KL(p,q): return float(np.sum(p*np.log(p/q)))
    return 0.5*KL(a,m) + 0.5*KL(b,m)

def _deg_counts(G, mode="total"):
    if mode == "in":
        vals = [d for _, d in G.in_degree()]
    elif mode == "out":
        vals = [d for _, d in G.out_degree()]
    else:
        vals = [G.in_degree(n)+G.out_degree(n) for n in G.nodes()]
    if not vals: return [1.0]
    c = Counter(vals)
    return [c.get(k,0) for k in range(max(c)+1)]

def _UG(G):
    UG = nx.Graph()
    UG.add_nodes_from(G.nodes(data=True))
    for u,v in G.edges():
        if u!=v: UG.add_edge(u,v)
    return UG

def _directed_density(G):
    n, m = G.number_of_nodes(), G.number_of_edges()
    return 0.0 if n<=1 else m/(n*(n-1))

def _avg_path_len_and_diam(UG, min_cc_size=2):
    comps = [UG.subgraph(c).copy() for c in nx.connected_components(UG)]
    comps = [C for C in comps if C.number_of_nodes()>=min_cc_size]
    if not comps:
        return (np.nan, np.nan, 0, [])
    sizes = np.array([C.number_of_nodes() for C in comps], dtype=float)
    w = sizes/sizes.sum()
    apls, diams = [], []
    for C in comps:
        if C.number_of_edges()==0 or C.number_of_nodes()==1:
            apls.append(np.inf); diams.append(0)
        else:
            apls.append(nx.average_shortest_path_length(C))
            diams.append(nx.diameter(C))
    apl = float(np.sum(w*np.array(apls)))
    Dmax = int(np.max(diams)) if len(diams) else np.nan
    return (apl, Dmax, int(len(comps)), sizes.tolist())

def _mean_clustering(UG):
    if UG.number_of_nodes()==0: return np.nan
    c = nx.clustering(UG)
    return float(np.mean(list(c.values()))) if c else np.nan

def _assort_in_in(G):
    try:
        return nx.degree_assortativity_coefficient(G, x='in', y='in')
    except Exception:
        return np.nan
def _assort_out_out(G):
    try:
        return nx.degree_assortativity_coefficient(G, x='out', y='out')
    except Exception:
        return np.nan
def _assort_out_in(G):
    try:
        return nx.degree_assortativity_coefficient(G, x='out', y='in')
    except Exception:
        return np.nan
def _assort_in_out(G):
    try:
        return nx.degree_assortativity_coefficient(G, x='in', y='out')
    except Exception:
        return np.nan

def _reciprocity(G):
    try:
        return nx.reciprocity(G)
    except Exception:
        return np.nan

def _spearman(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    if len(x)!=len(y) or len(x)==0: return np.nan
    # rank (dense-ties average)
    def rank(a):
        order = np.argsort(a, kind='mergesort')
        r = np.empty_like(order, dtype=float)
        r[order] = np.arange(len(a), dtype=float)
        uniq, starts = np.unique(a[order], return_index=True)
        for i,s in enumerate(starts):
            e = starts[i+1] if i+1<len(starts) else len(a)
            r[s:e] = np.mean(np.arange(s,e))
        inv = np.empty_like(order); inv[order] = np.arange(len(a))
        return r[inv]
    rx, ry = rank(x), rank(y)
    vx, vy = rx-rx.mean(), ry-ry.mean()
    denom = np.linalg.norm(vx)*np.linalg.norm(vy)
    return float(np.dot(vx,vy)/denom) if denom>0 else np.nan

def _broadcast_by_seed(G, seed_attr='status', cat_attr='category'):
    N = G.number_of_nodes()
    outdeg = dict(G.out_degree())
    cat = nx.get_node_attributes(G, cat_attr)
    is_seed = {n: str(G.nodes[n].get(seed_attr,'')).lower().startswith('seed') for n in G.nodes()}
    B = {}
    for s in G.nodes():
        if not is_seed[s]: continue
        kout = outdeg.get(s,0)
        if kout==0 or N<=1: B[s]=0.0; continue
        tgs = list(G.successors(s))
        if not tgs: B[s]=0.0; continue
        cats = [cat.get(t) for t in tgs]
        counts = Counter(cats); p = np.array([counts[c]/len(tgs) for c in counts])
        D = 1.0 - float(np.sum(p*p))
        B[s] = (kout/(N-1))*D
    return B

def _gatekeeper_all(G, seed_attr='status', cat_attr='category'):
    seeds = [n for n in G.nodes() if str(G.nodes[n].get(seed_attr,'')).lower().startswith('seed')]
    S = max(1, len(seeds))
    indeg = dict(G.in_degree())
    cat = nx.get_node_attributes(G, cat_attr)
    GK = {}
    for t in G.nodes():
        kin = indeg.get(t,0)
        if kin==0: GK[t]=0.0; continue
        src = list(G.predecessors(t))
        cats = [cat.get(s) for s in src]
        counts = Counter(cats); p = np.array([counts[c]/len(src) for c in counts])
        D = 1.0 - float(np.sum(p*p))
        GK[t] = (kin/S)*D
    return GK

def _directed_assortativities(G):
    f = nx.degree_pearson_correlation_coefficient
    return {
        "assort_out_in":  float(f(G, x="out", y="in")),
        "assort_in_out":  float(f(G, x="in",  y="out")),
        "assort_in_in":   float(f(G, x="in",  y="in")),
        "assort_out_out": float(f(G, x="out", y="out")),
    }



def _directed_louvain_fallback(G, nodes):
    print("AAAAAAAAAAAAAAAA")
    try:
        coms = list(nx.algorithms.community.greedy_modularity_communities(G.to_undirected()))
    except Exception:
        coms = [set(nodes)]
    labels = {}
    for idx, comm in enumerate(coms):
        for n in comm:
            labels[n] = idx
    next_id = max(labels.values(), default=-1) + 1
    for n in nodes:
        if n not in labels:
            labels[n] = next_id
            coms.append({n})
            next_id += 1
    print("Using directed Louvain fallback (greedy modularity on undirected).")
    return coms, labels, np.nan



def _mixing_matrices_full(G, category_attr="category"):
    """
    Ritorna:
      counts:   DataFrame (source_cat x target_cat)
      row_norm: counts normalizzato per riga
      col_norm: counts normalizzato per colonna
      cats_all: lista ordinata delle categorie presenti in righe o colonne
    """
    rows = []
    for u, v in G.edges():
        cu = G.nodes[u].get(category_attr, None)
        cv = G.nodes[v].get(category_attr, None)
        if cu is not None and cv is not None:
            rows.append((cu, cv))
    if not rows:
        empty = pd.DataFrame()
        return empty, empty, empty, []
    df = pd.DataFrame(rows, columns=["source_cat", "target_cat"])
    counts = pd.crosstab(df["source_cat"], df["target_cat"])
    # assicurati indici/colonne stringhe per allineamenti robusti
    counts.index = counts.index.astype(str); counts.columns = counts.columns.astype(str)
    with np.errstate(invalid='ignore', divide='ignore'):
        row_norm = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
        col_norm = counts.div(counts.sum(axis=0).replace(0, np.nan), axis=1)
    row_norm = row_norm.fillna(0.0); col_norm = col_norm.fillna(0.0)
    cats_all = sorted(set(counts.index).union(counts.columns))
    # completa con categorie mancanti (zero) per consistenza cross-layer
    counts = counts.reindex(index=cats_all, columns=cats_all, fill_value=0)
    row_norm = row_norm.reindex(index=cats_all, columns=cats_all, fill_value=0.0)
    col_norm = col_norm.reindex(index=cats_all, columns=cats_all, fill_value=0.0)
    return counts, row_norm, col_norm, cats_all

def _attribute_assortativity_category(UG, category_attr="category"):
    try:
        return nx.attribute_assortativity_coefficient(UG, category_attr)
    except Exception:
        return np.nan

def _flatten_mat(df):
    """Ritorna un vettore 1D flatten (row-major) come np.array float."""
    if df is None or df.size == 0: 
        return np.array([1.0])  # evitiamo nan nella JS
    return df.to_numpy(dtype=float).ravel()

def _nmi_from_partitions(part_H, part_T, nodes_common):
    """
    part_*: lista di insiemi (comunità) oppure dict node->comm
    nodes_common: lista/iter dei nodi sui quali calcolare NMI
    """
    # normalizza a dict node->label
    def to_map(part):
        if isinstance(part, dict):
            return part
        else:
            mapping = {}
            for i, comm in enumerate(part):
                for n in comm:
                    mapping[n] = i
            return mapping
    mH, mT = to_map(part_H), to_map(part_T)
    labsH = [mH.get(n, None) for n in nodes_common]
    labsT = [mT.get(n, None) for n in nodes_common]
    # filtra nodi senza etichetta in uno dei due
    filt = [(h,t) for h,t in zip(labsH, labsT) if h is not None and t is not None]
    if not filt:
        return np.nan
    labsH, labsT = zip(*filt)
    labsH = np.array(labsH); labsT = np.array(labsT)

    # costruisci contingenza
    uh = np.unique(labsH); ut = np.unique(labsT)
    idxH = {v:i for i,v in enumerate(uh)}
    idxT = {v:i for i,v in enumerate(ut)}
    C = np.zeros((len(uh), len(ut)), dtype=float)
    for h,t in zip(labsH, labsT):
        C[idxH[h], idxT[t]] += 1.0
    if C.sum() == 0:
        return np.nan
    P = C / C.sum()
    PH = P.sum(axis=1, keepdims=True)
    PT = P.sum(axis=0, keepdims=True)

    # entropie e MI
    def H(p):
        q = p[p>0]
        return -float(np.sum(q*np.log(q)))
    H_H = H(PH)
    H_T = H(PT)
    # mutual information: sum P_ij log(P_ij/(PH_i PT_j))
    mask = P>0
    MI = float(np.sum(P[mask]*np.log(P[mask]/(PH.dot(PT)[mask]))))
    denom = 0.5*(H_H + H_T)
    return MI/denom if denom>0 else np.nan

def compare_layers(
    G_h, G_t,
    use_common_nodes_only=True,
    min_cc_size=2,
    seed_attr='status',
    cat_attr='category',
    include_ratio_row=True
):
    """
    Output:
      per_layer_df: righe = ['H','T', ('H_over_T')] con metriche per-layer
                    + communities (oggetto), modularità, assortatività categoriale,
                      mixing matrices (oggetti: counts/row_norm/col_norm)
      cross_df:     una riga con metriche miste (JS, Spearman, Jaccard seed–seed & all,
                    corr. broadcast/GK, NMI comunità, JS mixing row/col)
    """
    # allineamento nodi
    if use_common_nodes_only:
        common = set(G_h.nodes()).intersection(G_t.nodes())
        H = G_h.subgraph(common).copy()
        T = G_t.subgraph(common).copy()
    else:
        H, T = G_h.copy(), G_t.copy()
        common = set(H.nodes()).intersection(T.nodes())
    common = sorted(common)

    # --- Per-layer (diretto)
    def dir_metrics(G):
        return dict(
            n=G.number_of_nodes(),
            m=G.number_of_edges(),
            density=_directed_density(G),
            reciprocity=_reciprocity(G),
            assort_out_out=_assort_out_out(G),
            assort_in_out=_assort_in_out(G),
            assort_in_in=_assort_in_in(G),
            assort_out_in=_assort_out_in(G)
        )
    H_dir = dir_metrics(H); T_dir = dir_metrics(T)

    # --- Per-layer (undirected)
    UH, UT = _UG(H), _UG(T)
    cl_H = _mean_clustering(UH); cl_T = _mean_clustering(UT)
    apl_H, D_H, ncc_H, sizes_H = _avg_path_len_and_diam(UH, min_cc_size=min_cc_size)
    apl_T, D_T, ncc_T, sizes_T = _avg_path_len_and_diam(UT, min_cc_size=min_cc_size)

    # --- Comunità (Directed Louvain) + modularità Q
    com_H, partH_map, _ = _run_directed_louvain(UH, weight_attr='weight', directed=False, seed=None)
    com_T, partT_map, _ = _run_directed_louvain(UT, weight_attr='weight', directed=False, seed=None)
    try:
        Q_H = nx.algorithms.community.quality.modularity(UH, com_H, weight='weight') if com_H else np.nan
    except Exception:
        Q_H = np.nan
    try:
        Q_T = nx.algorithms.community.quality.modularity(UT, com_T, weight='weight') if com_T else np.nan
    except Exception:
        Q_T = np.nan

    # --- Assortatività categoriale (undirected)
    cat_assort_H = _attribute_assortativity_category(UH, cat_attr)
    cat_assort_T = _attribute_assortativity_category(UT, cat_attr)

    # --- Mixing matrices categoria–categoria
    counts_H, rowH, colH, catsH = _mixing_matrices_full(H, category_attr=cat_attr)
    counts_T, rowT, colT, catsT = _mixing_matrices_full(T, category_attr=cat_attr)

    # costruisci per-layer DF
    per_layer_rows = []
    per_layer_rows.append(dict(
        layer='H',
        **H_dir,
        avg_clustering=cl_H, avg_path_len=apl_H,
        diameter_max=D_H, num_components=ncc_H,
        communities=com_H, modularity_Q=Q_H, n_communities=len(com_H),
        attr_assort_category=cat_assort_H,
        mixing_counts=counts_H, mixing_row_norm=rowH, mixing_col_norm=colH
    ))
    per_layer_rows.append(dict(
        layer='T',
        **T_dir,
        avg_clustering=cl_T, avg_path_len=apl_T,
        diameter_max=D_T, num_components=ncc_T,
        communities=com_T, modularity_Q=Q_T, n_communities=len(com_T),
        attr_assort_category=cat_assort_T,
        mixing_counts=counts_T, mixing_row_norm=rowT, mixing_col_norm=colT
    ))
    per_layer_df = pd.DataFrame(per_layer_rows).set_index('layer')

    if include_ratio_row:
        def r(a,b): 
            try:
                return (a/b) if (b not in (0,None) and not (isinstance(b,float) and np.isnan(b))) else np.nan
            except Exception:
                return np.nan
        ratio = {
            'n': r(per_layer_df.loc['H','n'], per_layer_df.loc['T','n']),
            'm': r(per_layer_df.loc['H','m'], per_layer_df.loc['T','m']),
            'density': r(per_layer_df.loc['H','density'], per_layer_df.loc['T','density']),
            'reciprocity': r(per_layer_df.loc['H','reciprocity'], per_layer_df.loc['T','reciprocity']),
            'assort_out_out': r(per_layer_df.loc['H','assort_out_out'], per_layer_df.loc['T','assort_out_out']),
            'assort_in_in': r(per_layer_df.loc['H','assort_in_in'], per_layer_df.loc['T','assort_in_in']),
            'assort_out_in': r(per_layer_df.loc['H','assort_out_in'], per_layer_df.loc['T','assort_out_in']),
            'assort_in_out': r(per_layer_df.loc['H','assort_in_out'], per_layer_df.loc['T','assort_in_out']),
            'avg_clustering': r(per_layer_df.loc['H','avg_clustering'], per_layer_df.loc['T','avg_clustering']),
            'avg_path_len': r(per_layer_df.loc['H','avg_path_len'], per_layer_df.loc['T','avg_path_len']),
            'diameter_max': r(per_layer_df.loc['H','diameter_max'], per_layer_df.loc['T','diameter_max']),
            'num_components': r(per_layer_df.loc['H','num_components'], per_layer_df.loc['T','num_components']),
            'modularity_Q': r(per_layer_df.loc['H','modularity_Q'], per_layer_df.loc['T','modularity_Q']),
            'n_communities': r(per_layer_df.loc['H','n_communities'], per_layer_df.loc['T','n_communities']),
            'attr_assort_category': r(per_layer_df.loc['H','attr_assort_category'], per_layer_df.loc['T','attr_assort_category']),
        }
        per_layer_df.loc['H_over_T'] = ratio

    # --- Metriche miste
    # JS tra distribuzioni di grado
    js_in   = _js_from_counts(_deg_counts(H,'in'),    _deg_counts(T,'in'))
    js_out  = _js_from_counts(_deg_counts(H,'out'),   _deg_counts(T,'out'))
    js_tot  = _js_from_counts(_deg_counts(H,'total'), _deg_counts(T,'total'))
    # Spearman tra vettori nodo-a-nodo (solo comuni)
    inH  = [H.in_degree(n)  for n in common]
    inT  = [T.in_degree(n)  for n in common]
    outH = [H.out_degree(n) for n in common]
    outT = [T.out_degree(n) for n in common]
    rho_in  = _spearman(inH, inT)
    rho_out = _spearman(outH, outT)
    # Coreness (undirected) e correlazione
    coreH = nx.core_number(UH) if UH.number_of_nodes()>0 else {}
    coreT = nx.core_number(UT) if UT.number_of_nodes()>0 else {}
    corH  = [coreH.get(n,0) for n in common]
    corT  = [coreT.get(n,0) for n in common]
    rho_core = _spearman(corH, corT)
    # Seed–seed Jaccard (archi diretti)
    def seed_flag(G, n): return str(G.nodes[n].get(seed_attr,'')).lower().startswith('seed')
    seedsH = {n for n in H.nodes() if seed_flag(H,n)}
    seedsT = {n for n in T.nodes() if seed_flag(T,n)}
    seedsC = seedsH & seedsT
    def E_seed(G, S):
        return {(u,v) for u,v in G.edges() if u in S and v in S and u!=v}
    EH, ET = E_seed(H,seedsC), E_seed(T,seedsC)
    jacc_seedseed = len(EH & ET)/len(EH | ET) if (len(EH|ET)>0) else np.nan

    # Edge Jaccard su tutti gli archi (non solo seed-seed)
    E_all_H = {(u,v) for u,v in H.edges() if u!=v}
    E_all_T = {(u,v) for u,v in T.edges() if u!=v}
    edge_jacc_all = len(E_all_H & E_all_T)/len(E_all_H | E_all_T) if (len(E_all_H|E_all_T)>0) else np.nan

    # Correlazioni broadcast (solo seed) e gatekeeper (tutti)
    B_H = _broadcast_by_seed(H, seed_attr=seed_attr, cat_attr=cat_attr)
    B_T = _broadcast_by_seed(T, seed_attr=seed_attr, cat_attr=cat_attr)
    seeds_BT = sorted(set(B_H.keys()) & set(B_T.keys()))
    rho_broadcast = _spearman([B_H[s] for s in seeds_BT], [B_T[s] for s in seeds_BT]) if seeds_BT else np.nan
    GK_H = _gatekeeper_all(H, seed_attr=seed_attr, cat_attr=cat_attr)
    GK_T = _gatekeeper_all(T, seed_attr=seed_attr, cat_attr=cat_attr)
    nodes_gk = sorted(set(GK_H.keys()) & set(GK_T.keys()))
    rho_gatekeeper = _spearman([GK_H[n] for n in nodes_gk], [GK_T[n] for n in nodes_gk]) if nodes_gk else np.nan

    # NMI tra comunità (undirected), sui nodi comuni
    nmi_comm = _nmi_from_partitions(com_H, com_T, common)

    # JS tra mixing row/col (flatten, allineando categorie)
    # unione categorie per allineare matrici
    cats_union = sorted(set(counts_H.index if hasattr(counts_H, 'index') else []).union(
                        set(counts_H.columns if hasattr(counts_H, 'columns') else [])).union(
                        set(counts_T.index if hasattr(counts_T, 'index') else [])).union(
                        set(counts_T.columns if hasattr(counts_T, 'columns') else [])))
    def reindex_square(df):
        if df is None or df.size==0:
            return pd.DataFrame(np.zeros((len(cats_union), len(cats_union))), index=cats_union, columns=cats_union)
        return df.reindex(index=cats_union, columns=cats_union, fill_value=0.0)
    rowH2 = reindex_square(rowH); rowT2 = reindex_square(rowT)
    colH2 = reindex_square(colH); colT2 = reindex_square(colT)
    js_mix_row = _js_from_counts(_flatten_mat(rowH2), _flatten_mat(rowT2))
    js_mix_col = _js_from_counts(_flatten_mat(colH2), _flatten_mat(colT2))

    cross_df = pd.DataFrame([dict(
        common_nodes=len(common),
        common_seeds=len(seedsC),
        js_in_degree=js_in,
        js_out_degree=js_out,
        js_total_degree=js_tot,
        spearman_in_degree=rho_in,
        spearman_out_degree=rho_out,
        spearman_coreness=rho_core,
        seedseed_jaccard=jacc_seedseed,
        edge_jaccard_all=edge_jacc_all,
        spearman_broadcast=rho_broadcast,
        spearman_gatekeeper=rho_gatekeeper,
        nmi_communities=nmi_comm,
        js_mixing_row=js_mix_row,
        js_mixing_col=js_mix_col
    )])

    return per_layer_df, cross_df.T

In [ ]:
def _run_directed_louvain(G, weight_attr="weight", directed=None, seed=42, resolution=1.0):
    """Run directed Louvain (Leiden) and return (communities, labels, modularity)."""
    directed = G.is_directed() if directed is None else bool(directed)
    nodes = list(G.nodes())
    if not nodes:
        return [], {}, np.nan
    if not _HAS_IGRAPH:
        warnings.warn("python-igraph unavailable; using greedy modularity fallback.", RuntimeWarning)
        return _directed_louvain_fallback(G, nodes)
    if directed and not _HAS_LEIDENALG:
        warnings.warn("leidenalg missing; directed Leiden unavailable. Falling back to greedy modularity.", RuntimeWarning)
        return _directed_louvain_fallback(G, nodes)
    node_index = {n: i for i, n in enumerate(nodes)}
    edges = []
    weights = []
    for u, v, data in G.edges(data=True):
        if u == v:
            continue
        w = float(data.get(weight_attr, 1.0))
        if np.isnan(w) or np.isinf(w):
            continue
        edges.append((node_index[u], node_index[v]))
        weights.append(w)
    if not edges:
        return _directed_louvain_fallback(G, nodes)
    g = ig.Graph(directed=directed)
    g.add_vertices(len(nodes))
    g.add_edges(edges)
    g.es["weight"] = weights if weights else [1.0] * len(edges)
    partition = None
    last_error = None
    rng_set = False
    try:
        if seed is not None:
            ig.set_random_number_generator(random.Random(int(seed) & 0xFFFFFFFF))
            rng_set = True
        if _HAS_LEIDENALG:
            try:
                partition = la.find_partition(
                    g,
                    la.RBConfigurationVertexPartition,
                    weights=g.es["weight"],
                    resolution_parameter=float(resolution),
                    n_iterations=-1
                )
            except Exception as err:
                last_error = err
                partition = None
        if partition is None and not directed:
            try:
                partition = g.community_leiden(
                    objective_function='modularity',
                    weights=g.es["weight"],
                    resolution=float(resolution),
                    n_iterations=-1
                )
            except TypeError as err:
                last_error = err
                try:
                    partition = g.community_leiden(
                        objective_function='modularity',
                        weights=g.es["weight"],
                        resolution_parameter=float(resolution),
                        n_iterations=-1
                    )
                except Exception as err2:
                    last_error = err2
                    partition = None
    except Exception as err:
        last_error = err
        partition = None
    finally:
        if rng_set:
            ig.set_random_number_generator(None)
    if partition is None:
        try:
            undirected = g.as_undirected()
            partition = undirected.community_multilevel(weights=undirected.es["weight"])
        except Exception as err:
            if last_error:
                warnings.warn(f"Leiden failed ({last_error}); using greedy modularity fallback.", RuntimeWarning)
            else:
                warnings.warn("Leiden failed without explicit error; using greedy modularity fallback.", RuntimeWarning)
            return _directed_louvain_fallback(G, nodes)
    membership = partition.membership
    modules = {}
    labels = {}
    for node, comm in zip(nodes, membership):
        modules.setdefault(comm, set()).add(node)
        labels[node] = comm
    modularity = np.nan
    if hasattr(partition, 'modularity') and partition.modularity is not None:
        modularity = partition.modularity
    else:
        try:
            modularity = g.modularity(membership, weights=g.es["weight"], directed=directed)
        except Exception:
            modularity = np.nan
    return list(modules.values()), labels, modularity

In [ ]:
# Similarity index from compare_layers output

def _as_cross_series(cross_df):
    """Convert either a row/column DataFrame or Series into a flat Series."""
    if isinstance(cross_df, pd.DataFrame):
        if cross_df.shape[0] == 1:
            return cross_df.iloc[0]
        if cross_df.shape[1] == 1:
            return cross_df.iloc[:, 0]
        raise ValueError("cross_df should have a single row or a single column.")
    if isinstance(cross_df, pd.Series):
        return cross_df
    raise TypeError("cross_df must be a pandas DataFrame or Series.")


def _js_to_sim(js_value, logbase='e'):
    """Map Jensen–Shannon divergence to a [0,1] similarity score (1 = identical)."""
    if js_value is None or (isinstance(js_value, float) and np.isnan(js_value)):
        return np.nan
    ln2 = np.log(2.0) if logbase == 'e' else 1.0
    return float(max(0.0, 1.0 - min(js_value, ln2) / ln2))


def _rho_to_sim(rho):
    """Rescale Spearman/Pearson correlation from [-1,1] into [0,1]."""
    if rho is None or (isinstance(rho, float) and np.isnan(rho)):
        return np.nan
    return float((rho + 1.0) / 2.0)


def _identity_sim(x):
    """Identity map for metrics already in [0,1] where 1 is best."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return float(np.clip(x, 0.0, 1.0))


def _weighted_mean(vals, wts):
    vals = np.asarray(vals, dtype=float)
    wts = np.asarray(wts, dtype=float)
    mask = ~np.isnan(vals) & ~np.isnan(wts) & (wts > 0)
    if not np.any(mask):
        return np.nan
    v = vals[mask]
    w = wts[mask]
    return float(np.sum(w * v) / np.sum(w))


def similarity_index_from_compare(
    per_layer_df,
    cross_df,
    use_metrics=None,
    weights=None,
    include_breakdown=True
):
    """Aggregate selected compare_layers metrics into a single similarity index."""
    cross = _as_cross_series(cross_df)
    if use_metrics is None:
        use_metrics = [
            "edge_jaccard_all",
            "seedseed_jaccard",
            "spearman_in_degree",
            "spearman_out_degree",
            "spearman_coreness",
            "nmi_communities",
            "js_in_degree",
            "js_out_degree",
            "js_mixing_row",
            "js_mixing_col",
        ]
    if weights is None:
        weights = {m: 1.0 for m in use_metrics}
    else:
        for m in use_metrics:
            weights.setdefault(m, 1.0)
    sims = {}
    raws = {}
    for metric in use_metrics:
        val = cross.get(metric, np.nan)
        raws[metric] = val
        if metric.startswith("js_"):
            sims[metric] = _js_to_sim(val)
        elif metric.startswith("spearman_"):
            sims[metric] = _rho_to_sim(val)
        elif metric in {"edge_jaccard_all", "seedseed_jaccard", "nmi_communities"}:
            sims[metric] = _identity_sim(val)
        else:
            if isinstance(val, (int, float)) and 0.0 <= val <= 1.0:
                sims[metric] = float(val)
            elif isinstance(val, (int, float)) and -1.0 <= val <= 1.0:
                sims[metric] = _rho_to_sim(val)
            else:
                sims[metric] = np.nan
    vals = [sims[m] for m in use_metrics]
    wts = [weights[m] for m in use_metrics]
    score = _weighted_mean(vals, wts)
    if not include_breakdown:
        return score
    mask = [
        not (isinstance(sims[m], float) and np.isnan(sims[m])) and weights[m] > 0
        for m in use_metrics
    ]
    wsum = sum(weights[m] for m, keep in zip(use_metrics, mask) if keep) or 1.0
    contribs = []
    for metric in use_metrics:
        w_norm = weights[metric] / wsum if wsum > 0 else 0.0
        sim = sims[metric]
        contribs.append(w_norm * sim if isinstance(sim, (int, float)) and not np.isnan(sim) else 0.0)
    breakdown = pd.DataFrame({
        "raw": [raws[m] for m in use_metrics],
        "similarity": [sims[m] for m in use_metrics],
        "weight": [weights[m] for m in use_metrics],
        "contribution": contribs,
    }, index=use_metrics)
    return score, breakdown

# Global Networks

In [ ]:
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')
h_edges = h_edges[h_edges['Source'] != h_edges['Target']]
h_edges = h_edges.groupby(['Source', 'Target']).size().reset_index(name='weight')
h_nodes.loc[27, 'Label'] = 'European Union'
# in h_nodes['STATUS'] change 'IN' to Seed and 'DISCOVERED' to 'Disc'
h_nodes['STATUS'] = h_nodes['STATUS'].replace({'IN': 'Seed', 'DISCOVERED': 'Disc'})
seeds_nodes = h_nodes[h_nodes['STATUS']=='Seed'].copy()
seeds_nodes.loc[:, 'Stance'] = [0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]
# Category mappings
id_to_cat_seeds = seeds_nodes.set_index('ID')['Category'].to_dict()
id_to_cat_h = h_nodes.set_index('ID')['Category'].to_dict()
# Create a directed graph for the Hyperlink data
G_h = nx.DiGraph()

for index, row in h_nodes.iterrows():
    G_h.add_node(row['ID'], label=row['Label'], status=row['STATUS'], category=row['Category'])

G_h.add_edges_from(h_edges[['Source', 'Target']].values)
#G_h.number_of_nodes(), G_h.number_of_edges(), nx.density(G_h)
import networkx as nx

def prune_by_in_degree(G, min_in=2):
    # proteggere i seed: non li filtra mai
    seeds = {n for n, d in G.nodes(data=True) if d.get('status') == 'Seed'}
    discovered = set(G.nodes()) - seeds

    keep_disc = {n for n in discovered if G.in_degree(n) >= min_in}
    keep = seeds | keep_disc
    H = G.subgraph(keep).copy()

    # rimuovi seed isolati dopo il pruning 
    iso_seeds = [n for n in seeds if H.degree(n) == 0]
    H.remove_nodes_from(iso_seeds)
    return H
G_h_p = prune_by_in_degree(G_h, min_in=2)

#G_h_p.number_of_nodes(), G_h_p.number_of_edges(), nx.density(G_h_p)
# --- load
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')

# porta dentro STATUS dai nodi hyperlink
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], on='ID', how='left')

# fallback: se STATUS mancante, metti 'DISCOVERED'
tw_nodes['STATUS'] = tw_nodes['STATUS'].fillna('DISCOVERED')

# rimuovi self-loops
tw_edges_clean = tw_edges[tw_edges['Source'] != tw_edges['Target']].copy()

# filtra solo retweet
#tw_edges_clean = tw_edges_clean[tw_edges_clean['Relationship'] == 'Retweet']

# costruisci pesi come numero di interazioni tra (Source, Target)
weighted_tw_edges = (
    tw_edges_clean
    .groupby(['Source', 'Target'], as_index=False)
    .size()
    .rename(columns={'size': 'weight'})
)
G_tw = nx.DiGraph()
for index, row in tw_nodes.iterrows():
    G_tw.add_node(row['ID'], label=row['Label'], status=row['STATUS'], category=row['Category'])
G_tw.add_weighted_edges_from(weighted_tw_edges[['Source', 'Target', 'weight']].values)
#G_tw.number_of_nodes(), G_tw.number_of_edges(), nx.density(G_tw)
def build_twitter_graphs(tw_nodes_df, weighted_edges_df, min_weight=1, return_weighted=True):
    """
    min_weight: soglia minima sul peso per tenere un arco (es. 3 -> almeno 3 interazioni)
    return_weighted: se True ritorna anche il grafo pesato coerente con il filtro
    """
    # filtra per soglia
    filt_edges = weighted_edges_df[weighted_edges_df['weight'] >= min_weight].copy()

    # --- Grafo binario filtrato
    G_bin = nx.DiGraph()
    for _, r in tw_nodes_df.iterrows():
        G_bin.add_node(r['ID'],
                       label=r.get('Label'),
                       status=r.get('STATUS'),
                       category=r.get('Category'))
    # aggiungi solo la coppia (u,v), ignorando il peso
    G_bin.add_edges_from(filt_edges[['Source', 'Target']].itertuples(index=False, name=None))

    if not return_weighted:
        return G_bin

    # --- Grafo pesato coerente con la soglia
    G_w = nx.DiGraph()
    for _, r in tw_nodes_df.iterrows():
        G_w.add_node(r['ID'],
                     label=r.get('Label'),
                     status=r.get('STATUS'),
                     category=r.get('Category'))
    for _, r in filt_edges.iterrows():
        G_w.add_edge(r['Source'], r['Target'], weight=int(r['weight']))

    return G_bin, G_w

# Esempi d'uso:
# (1) binario con tutti gli archi (min_weight=1)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=0, return_weighted=False)

# (2) binario + pesato, tenendo solo archi con almeno 3 interazioni
#G_tw_b3, G_tw_w3 = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=3, return_weighted=True)
G_tw_b.number_of_nodes(), G_tw_b.number_of_edges(), nx.density(G_tw_b)

# RACE

In [ ]:
import numpy as np
import networkx as nx

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def _rankdata(a):
    """
    Dense ranking with average for ties (like Spearman needs).
    Returns an array of ranks (float).
    """
    a = np.asarray(a, dtype=float)
    n = len(a)
    if n == 0:
        return a
    order = np.argsort(a, kind="mergesort")
    ranks = np.empty(n, dtype=float)
    ranks[order] = np.arange(n, dtype=float)
    # average ranks for ties
    vals, starts = np.unique(a[order], return_index=True)
    for i, s in enumerate(starts):
        e = starts[i+1] if i+1 < len(starts) else n
        ranks[s:e] = np.mean(np.arange(s, e))
    # undo ordering
    inv = np.empty_like(order)
    inv[order] = np.arange(n)
    return ranks[inv]


def _spearman_corr(x, y):
    """Spearman correlation without scipy."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) != len(y) or len(x) == 0:
        return np.nan
    rx = _rankdata(x)
    ry = _rankdata(y)
    vx = rx - rx.mean()
    vy = ry - ry.mean()
    denom = np.linalg.norm(vx) * np.linalg.norm(vy)
    if denom == 0:
        return np.nan
    return float(np.dot(vx, vy) / denom)


def _get_seed_flag_func(seed_attr="status"):
    """
    Returns a function is_seed(node, G) that checks if a node is a seed.
    Tries both 'status' and 'STATUS' attributes, plus startswith('seed').
    Adjust here if your attribute is named differently.
    """
    def is_seed(n, G):
        # try user-specified attribute
        val = G.nodes[n].get(seed_attr, None)
        if val is not None:
            s = str(val).strip().lower()
            # common cases: 'seed', 'in', 'seed_x', etc.
            return (s.startswith("seed")) or (s == "in")
        # fall back: STATUS (uppercase) if present
        if "STATUS" in G.nodes[n]:
            s = str(G.nodes[n]["STATUS"]).strip().lower()
            return (s.startswith("seed")) or (s == "in")
        return False
    return is_seed


# -------------------------------------------------------------------
# Edge Jaccard and Seed–seed Jaccard
# -------------------------------------------------------------------

def edge_jaccard(G1, G2):
    """
    Jaccard similarity between full directed edge sets of G1 and G2.
    Assumes nodes are already aligned (common node set).
    """
    E1 = set(G1.edges())
    E2 = set(G2.edges())
    if not E1 and not E2:
        return np.nan
    inter = len(E1 & E2)
    union = len(E1 | E2)
    return inter / union if union > 0 else np.nan


def seed_seed_jaccard(G1, G2, seed_attr="status"):
    """
    Jaccard similarity restricted to edges where both endpoints are seeds.
    Uses a flexible seed detection based on node attributes.
    """
    is_seed = _get_seed_flag_func(seed_attr)

    seeds1 = {n for n in G1.nodes() if is_seed(n, G1)}
    seeds2 = {n for n in G2.nodes() if is_seed(n, G2)}
    # work only on common seeds to be safe
    seedsC = seeds1 & seeds2

    def seed_edges(G):
        return {
            (u, v)
            for (u, v) in G.edges()
            if u in seedsC and v in seedsC and u != v
        }

    E1 = seed_edges(G1)
    E2 = seed_edges(G2)
    if not E1 and not E2:
        return np.nan
    inter = len(E1 & E2)
    union = len(E1 | E2)
    return inter / union if union > 0 else np.nan


# -------------------------------------------------------------------
# Spearman correlations on in-degree and out-degree
# -------------------------------------------------------------------

def degree_spearman(G1, G2):
    """
    Spearman correlations of in-degree and out-degree over common nodes.
    Returns (rho_in, rho_out).
    """
    nodes1 = set(G1.nodes())
    nodes2 = set(G2.nodes())
    common = sorted(nodes1 & nodes2)
    if not common:
        return np.nan, np.nan

    in1  = [G1.in_degree(n)  for n in common]
    in2  = [G2.in_degree(n)  for n in common]
    out1 = [G1.out_degree(n) for n in common]
    out2 = [G2.out_degree(n) for n in common]

    rho_in  = _spearman_corr(in1,  in2)
    rho_out = _spearman_corr(out1, out2)
    return rho_in, rho_out

def degree_pearson(G1, G2):
    """
    Pearrson correlations of in-degree and out-degree over common nodes.
    Returns (rho_in, rho_out).
    """
    nodes1 = set(G1.nodes())
    nodes2 = set(G2.nodes())
    common = sorted(nodes1 & nodes2)
    if not common:
        return np.nan, np.nan

    in1  = [G1.in_degree(n)  for n in common]
    in2  = [G2.in_degree(n)  for n in common]
    out1 = [G1.out_degree(n) for n in common]
    out2 = [G2.out_degree(n) for n in common]

    rho_in  = np.corrcoef(in1,  in2)[0, 1] if len(in1) > 1 else np.nan
    rho_out = np.corrcoef(out1, out2)[0, 1] if len(out1) > 1 else np.nan
    return rho_in, rho_out



# -------------------------------------------------------------------
# Community detection + NMI
# -------------------------------------------------------------------

def _directed_leiden_partition(G, weight_attr="weight", resolution=1.0, seed=42):
    """
    Try to run directed Leiden via python-igraph + leidenalg.
    Returns labels dict {node: community_id} or None if it fails.
    """
    try:
        import igraph as ig
        import leidenalg as la
    except Exception:
        return None

    nodes = list(G.nodes())
    if not nodes:
        return {}

    node_index = {n: i for i, n in enumerate(nodes)}
    edges = []
    weights = []
    for u, v, data in G.edges(data=True):
        if u == v:
            continue
        w = float(data.get(weight_attr, 1.0))
        if not (np.isnan(w) or np.isinf(w)):
            edges.append((node_index[u], node_index[v]))
            weights.append(w)

    if not edges:
        # trivial partition: all in the same community
        return {n: 0 for n in nodes}

    g = ig.Graph(directed=True)
    g.add_vertices(len(nodes))
    g.add_edges(edges)
    g.es["weight"] = weights

    # run Leiden with directed RBConfigurationVertexPartition
    if seed is not None:
        import random
        ig.set_random_number_generator(random.Random(int(seed) & 0xFFFFFFFF))
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights=g.es["weight"],
        resolution_parameter=float(resolution),
        n_iterations=-1,
    )
    membership = part.membership
    labels = {node: comm for node, comm in zip(nodes, membership)}
    return labels


def _undirected_louvain_fallback(G, weight_attr="weight"):
    """
    Fallback: undirected community detection (greedy modularity on NetworkX).
    Returns labels dict {node: community_id}.
    """
    import networkx.algorithms.community as nx_comm

    if G.number_of_nodes() == 0:
        return {}

    Gu = G.to_undirected()
    # greedy_modularity_communities ignores weights by default, which is fine as fallback
    coms = list(nx_comm.greedy_modularity_communities(Gu))
    labels = {}
    for cid, C in enumerate(coms):
        for n in C:
            labels[n] = cid
    return labels


def _compute_labels_for_nmi(G, weight_attr="weight", resolution=1.0):
    """
    Try directed Leiden; if not available, fall back to undirected greedy modularity.
    """
    labels = _directed_leiden_partition(G, weight_attr=weight_attr, resolution=resolution)
    if labels is None:
        labels = _undirected_louvain_fallback(G, weight_attr=weight_attr)
    return labels


def nmi_communities(G1, G2, weight_attr="weight", resolution=1.0):
    """
    Compute NMI between community partitions on the common nodes of G1 and G2.
    Uses sklearn.metrics.normalized_mutual_info_score if available;
    returns np.nan if sklearn is not installed.
    """
    try:
        from sklearn.metrics import normalized_mutual_info_score as nmi_score
    except Exception:
        return np.nan

    labels1 = _compute_labels_for_nmi(G1, weight_attr=weight_attr, resolution=resolution)
    labels2 = _compute_labels_for_nmi(G2, weight_attr=weight_attr, resolution=resolution)

    common = sorted(set(labels1.keys()) & set(labels2.keys()))
    if not common:
        return np.nan

    c1 = [labels1[n] for n in common]
    c2 = [labels2[n] for n in common]
    return float(nmi_score(c1, c2))


# -------------------------------------------------------------------
# Main wrapper
# -------------------------------------------------------------------

def compute_layer_similarity(G_h_common,
                             G_tw_b_common,
                             seed_attr="status",
                             weight_attr="weight",
                             leiden_resolution=1.0):
    """
    Compute the four similarity quantities between the two aligned graphs:
    - edge Jaccard
    - seed–seed Jaccard
    - Spearman in-degree, out-degree
    - NMI (community similarity, directed Leiden if available)

    Returns a dict.
    """

    # (Optional) sanity check: same node set
    nodes_h = set(G_h_common.nodes())
    nodes_t = set(G_tw_b_common.nodes())
    common_nodes = nodes_h & nodes_t
    if common_nodes != nodes_h or common_nodes != nodes_t:
        print("Warning: G_h_common and G_tw_b_common do not have exactly the same node set.")

    ej = edge_jaccard(G_h_common, G_tw_b_common)
    ssj = seed_seed_jaccard(G_h_common, G_tw_b_common, seed_attr=seed_attr)
    rho_in, rho_out = degree_spearman(G_h_common, G_tw_b_common)
    ro_in, ro_out = degree_pearson(G_h_common, G_tw_b_common)
    nmi = nmi_communities(G_h_common, G_tw_b_common,
                          weight_attr=weight_attr,
                          resolution=leiden_resolution)

    density_ratio = (nx.density(G_h_common) / nx.density(G_tw_b_common)) if nx.density(G_tw_b_common) > 0 else np.nan
    

    return {
        "edge_jaccard": ej,
        "seedseed_jaccard": ssj,
        #"spearman_in_degree": rho_in,
        #"spearman_out_degree": rho_out,
        "pearson_in_degree": ro_in,
        "pearson_out_degree": ro_out,
        "nmi_communities": nmi,
        #"density_ratio": density_ratio
    }


# -------------------------------------------------------------------
# Example usage (comment out in the notebook if not needed)
# -------------------------------------------------------------------
# results = compute_layer_similarity(G_h_common, G_tw_b_common,
#                                    seed_attr="status",  # or "STATUS"
#                                    weight_attr="weight",
#                                    leiden_resolution=0.27)
# print(results)

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects

def compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0):
    """
    Parameters
    ----------
    layer_graphs : list of networkx.Graph or DiGraph
        List of layers for the multiplex. All graphs must share the same node set
        (or at least their union is the multiplex node set).
    hub_z_threshold : float
        Threshold on z(o_i) to define hubs (default = 2.0 as in Battiston et al.)

    Returns
    -------
    results : dict
        keys: node
        values: dict with keys:
            'P'      : participation coefficient P_i
            'o'      : overlapping degree o_i
            'z_o'    : z-score of o_i
            'k_layer': dict {layer_index: k_i^[alpha]}
            'role'   : string, one of:
                       {'focused_hub', 'mixed_hub', 'multiplex_hub',
                        'focused_nonhub', 'mixed_nonhub', 'multiplex_nonhub'}
    """
    M = len(layer_graphs)
    # Union of all nodes across layers
    nodes = sorted(set().union(*[G.nodes() for G in layer_graphs]))

    # degrees per layer
    k_layer = {i: {} for i in nodes}
    for alpha, G in enumerate(layer_graphs):
        # use undirected degree; if graph is directed, use total degree or customize here
        deg_dict = dict(G.degree())
        for i in nodes:
            k_layer[i][alpha] = deg_dict.get(i, 0)

    # overlapping degree o_i = sum_α k_i^[α]
    o = {}
    for i in nodes:
        o[i] = sum(k_layer[i].values())

    # mean and std of overlapping degree
    o_vals = np.array(list(o.values()), dtype=float)
    o_mean = o_vals.mean()
    o_std = o_vals.std(ddof=0)

    # z-score of overlapping degree
    z_o = {}
    if o_std == 0:
        # all nodes have same overlapping degree; z-scores would all be zero
        for i in nodes:
            z_o[i] = 0.0
    else:
        for i in nodes:
            z_o[i] = (o[i] - o_mean) / o_std

    # participation coefficient P_i
    P = {}
    for i in nodes:
        if o[i] == 0:
            # isolated node in the multiplex; by convention P_i = 0
            P[i] = 0.0
            continue

        frac = np.array([k_layer[i][alpha] / o[i] for alpha in range(M)], dtype=float)
        P_i = (M / (M - 1.0)) * (1.0 - np.sum(frac ** 2))
        P[i] = P_i

    # classify nodes into 6 classes (focused/mixed/multiplex x hub/non-hub)
    def classify_node(i):
        Pi = P[i]
        zoi = z_o[i]

        if Pi <= 1.0 / 3.0:
            p_class = "focused"
        elif Pi <= 2.0 / 3.0:
            p_class = "mixed"
        else:
            p_class = "multiplex"

        if zoi >= hub_z_threshold:
            h_class = "hub"
        else:
            h_class = "nonhub"

        return f"{p_class}_{h_class}"

    results = {}
    for i in nodes:
        results[i] = {
            "P": P[i],
            "o": o[i],
            "z_o": z_o[i],
            "k_layer": k_layer[i],
            "role": classify_node(i),
        }

    return results

def plot_cartography(results, layer_graphs=None, title="Multiplex cartography: $P_i$ vs $z(o_i)$"):
    """
    Make a scatter plot like Battiston et al. Fig. 3(b): P_i vs z(o_i),
    using filled dots colored by node category. Seed nodes are annotated by ID.
    """
    # collect category/status info from the provided layers
    category_lookup = {}
    seed_nodes = set()
    if layer_graphs is not None:
        for G in layer_graphs:
            for node in results.keys():
                if node not in G:
                    continue
                data = G.nodes[node]
                category_lookup.setdefault(node, data.get('category') or data.get('Category'))
                if data.get('status') == 'Seed':
                    seed_nodes.add(node)

    default_cat_map = globals().get('id_to_cat_h', {})

    default_category_palette = {
        'ASN': 'red',
        'NGO': 'yellow',
        'GOV': 'green',
        'OTHER': 'blue',
        'PEC': 'black',
        'ROE': 'orange',
        'UNKNOWN': '#888888'
    }
    category_colors = dict(default_category_palette)
    category_colors.update(globals().get('color_map_cat', {}))
    category_colors = {str(k).upper(): v for k, v in category_colors.items()}

    def _normalize_category(cat):
        if cat is None or (isinstance(cat, float) and np.isnan(cat)):
            return 'UNKNOWN'
        cat_str = str(cat).strip()
        if not cat_str:
            return 'UNKNOWN'
        return cat_str.upper()

    # group nodes by category for colored scatter
    grouped = {}
    for node, vals in results.items():
        raw_cat = category_lookup.get(node) or default_cat_map.get(node)
        cat_label = _normalize_category(raw_cat)
        grouped.setdefault(cat_label, {"P": [], "z_o": [], "nodes": []})
        grouped[cat_label]["P"].append(vals["P"])
        grouped[cat_label]["z_o"].append(vals["z_o"])
        grouped[cat_label]["nodes"].append(node)

    plt.figure(figsize=(9, 6))

    handles = []
    for cat_label, payload in grouped.items():
        color = category_colors.get(cat_label, category_colors['UNKNOWN'])
        legend_label = cat_label if cat_label != 'UNKNOWN' else 'Unknown'
        sc = plt.scatter(
            payload['P'],
            payload['z_o'],
            label=legend_label,
            color=color,
            edgecolors='black',
            s=320,
            alpha=0.85
        )
        handles.append(sc)

    # region separators at P = 1/3 and 2/3
    plt.axvline(1.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    plt.axvline(2.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    # hub threshold z = 2
    plt.axhline(2.0, color="gray", linestyle=":", linewidth=1)

    # annotate seed nodes with their ID centered on the dot
    for node in sorted(seed_nodes):
        if node not in results:
            continue
        Pi = results[node]["P"]
        zoi = results[node]["z_o"]
        txt = plt.text(Pi, zoi, str(node), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        txt.set_path_effects([PathEffects.withStroke(linewidth=2.2, foreground='black')])

    plt.xlabel(r"Participation coefficient $P_i$")
    plt.ylabel(r"$z(o_i)$ (z-score of overlapping degree)")
    #plt.title(title)
    if handles:
        plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()


# === Usage with your two layers ============================================
# assuming:
#   G_h_common      : networkx.Graph for Hyperlink layer
#   G_tw_b_common   : networkx.Graph for Twitter layer (binary, common nodes)

In [ ]:
G_h_p = prune_by_in_degree(G_h, min_in=0)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=0, return_weighted=False)
common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())

# Build induced subgraphs containing only common nodes
G_h_common = G_h_p.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

print(G_h_common.number_of_nodes(), G_h_common.number_of_edges())
print(G_tw_b_common.number_of_nodes(), G_tw_b_common.number_of_edges())



In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects

def compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0):
    """
    Parameters
    ----------
    layer_graphs : list of networkx.Graph or DiGraph
        List of layers for the multiplex. All graphs must share the same node set
        (or at least their union is the multiplex node set).
    hub_z_threshold : float
        Threshold on z(o_i) to define hubs (default = 2.0 as in Battiston et al.)

    Returns
    -------
    results : dict
        keys: node
        values: dict with keys:
            'P'      : participation coefficient P_i
            'o'      : overlapping degree o_i
            'z_o'    : z-score of o_i
            'k_layer': dict {layer_index: k_i^[alpha]}
            'role'   : string, one of:
                       {'focused_hub', 'mixed_hub', 'multiplex_hub',
                        'focused_nonhub', 'mixed_nonhub', 'multiplex_nonhub'}
    """
    M = len(layer_graphs)
    # Union of all nodes across layers
    nodes = sorted(set().union(*[G.nodes() for G in layer_graphs]))

    # degrees per layer
    k_layer = {i: {} for i in nodes}
    for alpha, G in enumerate(layer_graphs):
        # use undirected degree; if graph is directed, use total degree or customize here
        deg_dict = dict(G.degree())
        for i in nodes:
            k_layer[i][alpha] = deg_dict.get(i, 0)

    # overlapping degree o_i = sum_α k_i^[α]
    o = {}
    for i in nodes:
        o[i] = sum(k_layer[i].values())

    # mean and std of overlapping degree
    o_vals = np.array(list(o.values()), dtype=float)
    o_mean = o_vals.mean()
    o_std = o_vals.std(ddof=0)

    # z-score of overlapping degree
    z_o = {}
    if o_std == 0:
        # all nodes have same overlapping degree; z-scores would all be zero
        for i in nodes:
            z_o[i] = 0.0
    else:
        for i in nodes:
            z_o[i] = (o[i] - o_mean) / o_std

    # participation coefficient P_i
    P = {}
    for i in nodes:
        if o[i] == 0:
            # isolated node in the multiplex; by convention P_i = 0
            P[i] = 0.0
            continue

        frac = np.array([k_layer[i][alpha] / o[i] for alpha in range(M)], dtype=float)
        P_i = (M / (M - 1.0)) * (1.0 - np.sum(frac ** 2))
        P[i] = P_i

    # classify nodes into 6 classes (focused/mixed/multiplex x hub/non-hub)
    def classify_node(i):
        Pi = P[i]
        zoi = z_o[i]

        if Pi <= 1.0 / 3.0:
            p_class = "focused"
        elif Pi <= 2.0 / 3.0:
            p_class = "mixed"
        else:
            p_class = "multiplex"

        if zoi >= hub_z_threshold:
            h_class = "hub"
        else:
            h_class = "nonhub"

        return f"{p_class}_{h_class}"

    results = {}
    for i in nodes:
        results[i] = {
            "P": P[i],
            "o": o[i],
            "z_o": z_o[i],
            "k_layer": k_layer[i],
            "role": classify_node(i),
        }

    return results

def plot_cartography(results, layer_graphs=None, title="Multiplex cartography: $P_i$ vs $z(o_i)$"):
    """
    Make a scatter plot like Battiston et al. Fig. 3(b): P_i vs z(o_i),
    using filled dots colored by node category. Seed nodes are annotated by ID.
    """
    # collect category/status info from the provided layers
    category_lookup = {}
    seed_nodes = set()
    if layer_graphs is not None:
        for G in layer_graphs:
            for node in results.keys():
                if node not in G:
                    continue
                data = G.nodes[node]
                category_lookup.setdefault(node, data.get('category') or data.get('Category'))
                if data.get('status') == 'Seed':
                    seed_nodes.add(node)

    default_cat_map = globals().get('id_to_cat_h', {})

    default_category_palette = {
        'ASN': 'red',
        'NGO': 'yellow',
        'GOV': 'green',
        'OTHER': 'blue',
        'PEC': 'black',
        'ROE': 'orange',
        'UNKNOWN': '#888888'
    }
    category_colors = dict(default_category_palette)
    category_colors.update(globals().get('color_map_cat', {}))
    category_colors = {str(k).upper(): v for k, v in category_colors.items()}

    forced_node_colors = {408: '#000000'}

    def _normalize_category(cat):
        if cat is None or (isinstance(cat, float) and np.isnan(cat)):
            return 'UNKNOWN'
        cat_str = str(cat).strip()
        if not cat_str:
            return 'UNKNOWN'
        return cat_str.upper()

    # group nodes by category for colored scatter (with per-node overrides)
    grouped = {}
    for node, vals in results.items():
        raw_cat = category_lookup.get(node) or default_cat_map.get(node)
        cat_label = _normalize_category(raw_cat)
        if node in forced_node_colors:
            group_key = f"forced_{node}"
            legend_label = None
            color = forced_node_colors[node]
        else:
            group_key = cat_label
            legend_label = cat_label if cat_label != 'UNKNOWN' else 'Unknown'
            color = category_colors.get(cat_label, category_colors['UNKNOWN'])
        grouped.setdefault(group_key, {
            "P": [],
            "z_o": [],
            "nodes": [],
            "color": color,
            "label": legend_label
        })
        grouped[group_key]["P"].append(vals["P"])
        grouped[group_key]["z_o"].append(vals["z_o"])
        grouped[group_key]["nodes"].append(node)

    plt.figure(figsize=(9, 6))

    handles = []
    for group_key, payload in grouped.items():
        sc = plt.scatter(
            payload['P'],
            payload['z_o'],
            label=payload['label'],
            color=payload['color'],
            edgecolors='black',
            s=320,
            alpha=0.85
        )
        if payload['label']:
            handles.append(sc)

    # region separators at P = 1/3 and 2/3
    plt.axvline(1.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    plt.axvline(2.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    # hub threshold z = 2
    plt.axhline(2.0, color="gray", linestyle=":", linewidth=1)

    # annotate seed nodes with their ID centered on the dot
    for node in sorted(seed_nodes):
        if node not in results:
            continue
        Pi = results[node]["P"]
        zoi = results[node]["z_o"]
        txt = plt.text(Pi, zoi, str(node), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        txt.set_path_effects([PathEffects.withStroke(linewidth=2.2, foreground='black')])

    plt.xlabel(r"Participation coefficient $P_i$")
    plt.ylabel(r"$z(o_i)$ (z-score of overlapping degree)")
    #plt.title(title)
    if handles:
        plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()



# === Usage with your two layers ============================================
# assuming:
#   G_h_common      : networkx.Graph for Hyperlink layer
#   G_tw_b_common   : networkx.Graph for Twitter layer (binary, common nodes)

if __name__ == "__main__":
    layer_graphs = [G_h_common, G_tw_b_common]  # order does not matter

    results = compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0)
    plot_cartography(results, layer_graphs=layer_graphs, title="Cartography for G_h_common + G_tw_b_common")

In [ ]:
G_h_p = prune_by_in_degree(G_h, min_in=0)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=4, return_weighted=False)
common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())

# Build induced subgraphs containing only common nodes
G_h_common = G_h_p.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

print(G_h_common.number_of_nodes(), G_h_common.number_of_edges())
print(G_tw_b_common.number_of_nodes(), G_tw_b_common.number_of_edges())



In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects

def compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0):
    """
    Parameters
    ----------
    layer_graphs : list of networkx.Graph or DiGraph
        List of layers for the multiplex. All graphs must share the same node set
        (or at least their union is the multiplex node set).
    hub_z_threshold : float
        Threshold on z(o_i) to define hubs (default = 2.0 as in Battiston et al.)

    Returns
    -------
    results : dict
        keys: node
        values: dict with keys:
            'P'      : participation coefficient P_i
            'o'      : overlapping degree o_i
            'z_o'    : z-score of o_i
            'k_layer': dict {layer_index: k_i^[alpha]}
            'role'   : string, one of:
                       {'focused_hub', 'mixed_hub', 'multiplex_hub',
                        'focused_nonhub', 'mixed_nonhub', 'multiplex_nonhub'}
    """
    M = len(layer_graphs)
    # Union of all nodes across layers
    nodes = sorted(set().union(*[G.nodes() for G in layer_graphs]))

    # degrees per layer
    k_layer = {i: {} for i in nodes}
    for alpha, G in enumerate(layer_graphs):
        # use undirected degree; if graph is directed, use total degree or customize here
        deg_dict = dict(G.degree())
        for i in nodes:
            k_layer[i][alpha] = deg_dict.get(i, 0)

    # overlapping degree o_i = sum_α k_i^[α]
    o = {}
    for i in nodes:
        o[i] = sum(k_layer[i].values())

    # mean and std of overlapping degree
    o_vals = np.array(list(o.values()), dtype=float)
    o_mean = o_vals.mean()
    o_std = o_vals.std(ddof=0)

    # z-score of overlapping degree
    z_o = {}
    if o_std == 0:
        # all nodes have same overlapping degree; z-scores would all be zero
        for i in nodes:
            z_o[i] = 0.0
    else:
        for i in nodes:
            z_o[i] = (o[i] - o_mean) / o_std

    # participation coefficient P_i
    P = {}
    for i in nodes:
        if o[i] == 0:
            # isolated node in the multiplex; by convention P_i = 0
            P[i] = 0.0
            continue

        frac = np.array([k_layer[i][alpha] / o[i] for alpha in range(M)], dtype=float)
        P_i = (M / (M - 1.0)) * (1.0 - np.sum(frac ** 2))
        P[i] = P_i

    # classify nodes into 6 classes (focused/mixed/multiplex x hub/non-hub)
    def classify_node(i):
        Pi = P[i]
        zoi = z_o[i]

        if Pi <= 1.0 / 3.0:
            p_class = "focused"
        elif Pi <= 2.0 / 3.0:
            p_class = "mixed"
        else:
            p_class = "multiplex"

        if zoi >= hub_z_threshold:
            h_class = "hub"
        else:
            h_class = "nonhub"

        return f"{p_class}_{h_class}"

    results = {}
    for i in nodes:
        results[i] = {
            "P": P[i],
            "o": o[i],
            "z_o": z_o[i],
            "k_layer": k_layer[i],
            "role": classify_node(i),
        }

    return results

def plot_cartography(results, layer_graphs=None, title="Multiplex cartography: $P_i$ vs $z(o_i)$"):
    """
    Make a scatter plot like Battiston et al. Fig. 3(b): P_i vs z(o_i),
    using filled dots colored by node category. Seed nodes are annotated by ID.
    """
    # collect category/status info from the provided layers
    category_lookup = {}
    seed_nodes = set()
    if layer_graphs is not None:
        for G in layer_graphs:
            for node in results.keys():
                if node not in G:
                    continue
                data = G.nodes[node]
                category_lookup.setdefault(node, data.get('category') or data.get('Category'))
                if data.get('status') == 'Seed':
                    seed_nodes.add(node)

    default_cat_map = globals().get('id_to_cat_h', {})

    default_category_palette = {
        'ASN': 'red',
        'NGO': 'yellow',
        'GOV': 'green',
        'OTHER': 'blue',
        'PEC': 'black',
        'ROE': 'orange',
        'UNKNOWN': '#888888'
    }
    category_colors = dict(default_category_palette)
    category_colors.update(globals().get('color_map_cat', {}))
    category_colors = {str(k).upper(): v for k, v in category_colors.items()}

    forced_node_colors = {408: '#000000'}

    def _normalize_category(cat):
        if cat is None or (isinstance(cat, float) and np.isnan(cat)):
            return 'UNKNOWN'
        cat_str = str(cat).strip()
        if not cat_str:
            return 'UNKNOWN'
        return cat_str.upper()

    # group nodes by category for colored scatter (with per-node overrides)
    grouped = {}
    for node, vals in results.items():
        raw_cat = category_lookup.get(node) or default_cat_map.get(node)
        cat_label = _normalize_category(raw_cat)
        if node in forced_node_colors:
            group_key = f"forced_{node}"
            legend_label = None
            color = forced_node_colors[node]
        else:
            group_key = cat_label
            legend_label = cat_label if cat_label != 'UNKNOWN' else 'Unknown'
            color = category_colors.get(cat_label, category_colors['UNKNOWN'])
        grouped.setdefault(group_key, {
            "P": [],
            "z_o": [],
            "nodes": [],
            "color": color,
            "label": legend_label
        })
        grouped[group_key]["P"].append(vals["P"])
        grouped[group_key]["z_o"].append(vals["z_o"])
        grouped[group_key]["nodes"].append(node)

    plt.figure(figsize=(9, 6))

    handles = []
    for group_key, payload in grouped.items():
        sc = plt.scatter(
            payload['P'],
            payload['z_o'],
            label=payload['label'],
            color=payload['color'],
            edgecolors='black',
            s=320,
            alpha=0.85
        )
        if payload['label']:
            handles.append(sc)

    # region separators at P = 1/3 and 2/3
    plt.axvline(1.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    plt.axvline(2.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    # hub threshold z = 2
    plt.axhline(2.0, color="gray", linestyle=":", linewidth=1)

    # annotate seed nodes with their ID centered on the dot
    for node in sorted(seed_nodes):
        if node not in results:
            continue
        Pi = results[node]["P"]
        zoi = results[node]["z_o"]
        txt = plt.text(Pi, zoi, str(node), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        txt.set_path_effects([PathEffects.withStroke(linewidth=2.2, foreground='black')])

    plt.xlabel(r"Participation coefficient $P_i$")
    plt.ylabel(r"$z(o_i)$ (z-score of overlapping degree)")
    #plt.title(title)
    if handles:
        plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()



# === Usage with your two layers ============================================
# assuming:
#   G_h_common      : networkx.Graph for Hyperlink layer
#   G_tw_b_common   : networkx.Graph for Twitter layer (binary, common nodes)

if __name__ == "__main__":
    layer_graphs = [G_h_common, G_tw_b_common]  # order does not matter

    results = compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0)
    plot_cartography(results, layer_graphs=layer_graphs, title="Cartography for G_h_common + G_tw_b_common")

In [ ]:
G_h_p = prune_by_in_degree(G_h, min_in=0)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=14, return_weighted=False)
common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())

# Build induced subgraphs containing only common nodes
G_h_common = G_h_p.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

print(G_h_common.number_of_nodes(), G_h_common.number_of_edges())
print(G_tw_b_common.number_of_nodes(), G_tw_b_common.number_of_edges())



In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects

def compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0):
    """
    Parameters
    ----------
    layer_graphs : list of networkx.Graph or DiGraph
        List of layers for the multiplex. All graphs must share the same node set
        (or at least their union is the multiplex node set).
    hub_z_threshold : float
        Threshold on z(o_i) to define hubs (default = 2.0 as in Battiston et al.)

    Returns
    -------
    results : dict
        keys: node
        values: dict with keys:
            'P'      : participation coefficient P_i
            'o'      : overlapping degree o_i
            'z_o'    : z-score of o_i
            'k_layer': dict {layer_index: k_i^[alpha]}
            'role'   : string, one of:
                       {'focused_hub', 'mixed_hub', 'multiplex_hub',
                        'focused_nonhub', 'mixed_nonhub', 'multiplex_nonhub'}
    """
    M = len(layer_graphs)
    # Union of all nodes across layers
    nodes = sorted(set().union(*[G.nodes() for G in layer_graphs]))

    # degrees per layer
    k_layer = {i: {} for i in nodes}
    for alpha, G in enumerate(layer_graphs):
        # use undirected degree; if graph is directed, use total degree or customize here
        deg_dict = dict(G.degree())
        for i in nodes:
            k_layer[i][alpha] = deg_dict.get(i, 0)

    # overlapping degree o_i = sum_α k_i^[α]
    o = {}
    for i in nodes:
        o[i] = sum(k_layer[i].values())

    # mean and std of overlapping degree
    o_vals = np.array(list(o.values()), dtype=float)
    o_mean = o_vals.mean()
    o_std = o_vals.std(ddof=0)

    # z-score of overlapping degree
    z_o = {}
    if o_std == 0:
        # all nodes have same overlapping degree; z-scores would all be zero
        for i in nodes:
            z_o[i] = 0.0
    else:
        for i in nodes:
            z_o[i] = (o[i] - o_mean) / o_std

    # participation coefficient P_i
    P = {}
    for i in nodes:
        if o[i] == 0:
            # isolated node in the multiplex; by convention P_i = 0
            P[i] = 0.0
            continue

        frac = np.array([k_layer[i][alpha] / o[i] for alpha in range(M)], dtype=float)
        P_i = (M / (M - 1.0)) * (1.0 - np.sum(frac ** 2))
        P[i] = P_i

    # classify nodes into 6 classes (focused/mixed/multiplex x hub/non-hub)
    def classify_node(i):
        Pi = P[i]
        zoi = z_o[i]

        if Pi <= 1.0 / 3.0:
            p_class = "focused"
        elif Pi <= 2.0 / 3.0:
            p_class = "mixed"
        else:
            p_class = "multiplex"

        if zoi >= hub_z_threshold:
            h_class = "hub"
        else:
            h_class = "nonhub"

        return f"{p_class}_{h_class}"

    results = {}
    for i in nodes:
        results[i] = {
            "P": P[i],
            "o": o[i],
            "z_o": z_o[i],
            "k_layer": k_layer[i],
            "role": classify_node(i),
        }

    return results

def plot_cartography(results, layer_graphs=None, title="Multiplex cartography: $P_i$ vs $z(o_i)$"):
    """
    Make a scatter plot like Battiston et al. Fig. 3(b): P_i vs z(o_i),
    using filled dots colored by node category. Seed nodes are annotated by ID.
    """
    # collect category/status info from the provided layers
    category_lookup = {}
    seed_nodes = set()
    if layer_graphs is not None:
        for G in layer_graphs:
            for node in results.keys():
                if node not in G:
                    continue
                data = G.nodes[node]
                category_lookup.setdefault(node, data.get('category') or data.get('Category'))
                if data.get('status') == 'Seed':
                    seed_nodes.add(node)

    default_cat_map = globals().get('id_to_cat_h', {})

    default_category_palette = {
        'ASN': 'red',
        'NGO': 'yellow',
        'GOV': 'green',
        'OTHER': 'blue',
        'PEC': 'black',
        'ROE': 'orange',
        'UNKNOWN': '#888888'
    }
    category_colors = dict(default_category_palette)
    category_colors.update(globals().get('color_map_cat', {}))
    category_colors = {str(k).upper(): v for k, v in category_colors.items()}

    forced_node_colors = {408: '#000000'}

    def _normalize_category(cat):
        if cat is None or (isinstance(cat, float) and np.isnan(cat)):
            return 'UNKNOWN'
        cat_str = str(cat).strip()
        if not cat_str:
            return 'UNKNOWN'
        return cat_str.upper()

    # group nodes by category for colored scatter (with per-node overrides)
    grouped = {}
    for node, vals in results.items():
        raw_cat = category_lookup.get(node) or default_cat_map.get(node)
        cat_label = _normalize_category(raw_cat)
        if node in forced_node_colors:
            group_key = f"forced_{node}"
            legend_label = None
            color = forced_node_colors[node]
        else:
            group_key = cat_label
            legend_label = cat_label if cat_label != 'UNKNOWN' else 'Unknown'
            color = category_colors.get(cat_label, category_colors['UNKNOWN'])
        grouped.setdefault(group_key, {
            "P": [],
            "z_o": [],
            "nodes": [],
            "color": color,
            "label": legend_label
        })
        grouped[group_key]["P"].append(vals["P"])
        grouped[group_key]["z_o"].append(vals["z_o"])
        grouped[group_key]["nodes"].append(node)

    plt.figure(figsize=(9, 6))

    handles = []
    for group_key, payload in grouped.items():
        sc = plt.scatter(
            payload['P'],
            payload['z_o'],
            label=payload['label'],
            color=payload['color'],
            edgecolors='black',
            s=320,
            alpha=0.85
        )
        if payload['label']:
            handles.append(sc)

    # region separators at P = 1/3 and 2/3
    plt.axvline(1.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    plt.axvline(2.0 / 3.0, color="gray", linestyle="--", linewidth=1)
    # hub threshold z = 2
    plt.axhline(2.0, color="gray", linestyle=":", linewidth=1)

    # annotate seed nodes with their ID centered on the dot
    for node in sorted(seed_nodes):
        if node not in results:
            continue
        Pi = results[node]["P"]
        zoi = results[node]["z_o"]
        txt = plt.text(Pi, zoi, str(node), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        txt.set_path_effects([PathEffects.withStroke(linewidth=2.2, foreground='black')])

    plt.xlabel(r"Participation coefficient $P_i$")
    plt.ylabel(r"$z(o_i)$ (z-score of overlapping degree)")
    #plt.title(title)
    if handles:
        plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()



# === Usage with your two layers ============================================
# assuming:
#   G_h_common      : networkx.Graph for Hyperlink layer
#   G_tw_b_common   : networkx.Graph for Twitter layer (binary, common nodes)

if __name__ == "__main__":
    layer_graphs = [G_h_common, G_tw_b_common]  # order does not matter

    results = compute_multiplex_cartography(layer_graphs, hub_z_threshold=2.0)
    plot_cartography(results, layer_graphs=layer_graphs, title="Cartography for G_h_common + G_tw_b_common")

In [ ]:
from unittest import result


results_per_threshold = {}

for min_weight_per_cell in range(41):

    G_h_p = prune_by_in_degree(G_h, min_in=0)
    G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=min_weight_per_cell, return_weighted=False)
    common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())
    # Build induced subgraphs containing only common nodes
    G_h_common = G_h_p.subgraph(common_nodes).copy()
    G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

    # take the LCC
    G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
    G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

    common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
    # Build induced subgraphs containing only common nodes
    G_h_common = G_h_common.subgraph(common_nodes).copy()
    G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()

    results = compute_layer_similarity(G_h_common, G_tw_b_common,
                                    seed_attr="status",
                                    weight_attr="weight",
                                    leiden_resolution=1.0)
    results_per_threshold[min_weight_per_cell] = results

# results_per_threshold to DataFrame
results_df = pd.DataFrame.from_dict(results_per_threshold, orient='index')
results_df.index.name = 'min_weight'
results_df.reset_index(inplace=True)
results_df

In [ ]:
# plot results_df
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
for col in results_df.columns:
    if col != 'min_weight':
        plt.plot(results_df['min_weight'], results_df[col], marker='o', label=col)
plt.xlabel('Minimum Weight (Twitter edges)')
plt.ylabel('Similarity Metric Value')
#plt.title('Layer Similarity Metrics vs Minimum Weight per Cell')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# plot results_df and the sum of all metric curves
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# original individual curves on ax1
for col in results_df.columns:
    if col != 'min_weight':
        ax1.plot(results_df['min_weight'],
                 results_df[col],
                 marker='o',
                 label=col,
                 alpha=0.8)

# second y-axis for avg of all metrics
ax2 = ax1.twinx()

metric_cols = [c for c in results_df.columns if c != 'min_weight']
sum_series = results_df[metric_cols].sum(axis=1)

ax2.plot(results_df['min_weight'],
         sum_series / len(metric_cols),
         marker='o',
         color='black',
         linewidth=2.0,
         label='avg_all_metrics')

ax2.set_ylim(0., .8)

ax1.set_xlabel('Minimum Weight (Twitter edges)')
ax1.set_ylabel('Individual Similarity Metrics')
ax2.set_ylabel('Average of All Metrics')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

# grid aligned with left (ax1) y-axis
ax1.grid(True)

plt.show()


In [ ]:
# plot results_df and the sum of all metric curves
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# original individual curves on ax1
for col in results_df.columns:
    if col != 'min_weight':
        ax1.plot(results_df['min_weight'],
                 results_df[col],
                 marker='o',
                 label=col,
                 alpha=0.8)

# second y-axis for avg of all metrics
ax2 = ax1.twinx()

metric_cols = [c for c in results_df.columns if c != 'min_weight']
sum_series = results_df[metric_cols].sum(axis=1)

ax2.plot(results_df['min_weight'],
         sum_series / len(metric_cols),
         marker='o',
         color='gray',
         linewidth=2.0,
         # tratteggiata
         linestyle = '--',
         label='avg_all_metrics')

ax2.set_ylim(0., .8)

ax1.set_xlabel('Minimum Weight (Twitter edges)')
ax1.set_ylabel('Individual Similarity Metrics')
ax2.set_ylabel('Average of All Metrics')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

# grid aligned with left (ax1) y-axis
ax1.grid(True)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

x = results_df['min_weight']
metric_cols = [c for c in results_df.columns if c != 'min_weight']

# 1) Curve individuali
for col in metric_cols:
    ax.plot(
        x,
        results_df[col],
        marker='.',
        linestyle='--',
        label=col,
        linewidth=1.6,
        alpha=0.8
    )

# 2) Media e deviazione standard
values = results_df[metric_cols].values
mean_series = values.mean(axis=1)
std_series = values.std(axis=1, ddof=1)  # deviazione standard campionaria

lower = mean_series - std_series
upper = mean_series + std_series

# linea della media (tratteggiata e grigia)
ax.plot(
    x,
    mean_series,
    marker = 'o',
    
    color='grey',
    linewidth=3.0,
    label='avg_all_metrics'
)


ax.set_xlabel('Minimum weight per edge in Twitter layer')
ax.set_ylabel('Similarity metrics (0–1)')
ax.set_ylim(0.0, 1.0)

# legenda unica
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='lower right')

ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

x = results_df['min_weight']
metric_cols = [c for c in results_df.columns if c != 'min_weight']

# 1) Curve individuali
for col in metric_cols:
    ax.plot(
        x,
        results_df[col],
        marker='.',
        linestyle='-',
        label=col,
        linewidth=1.2,
        alpha=0.8
    )

# 2) Media e deviazione standard
values = results_df[metric_cols].values
mean_series = values.mean(axis=1)
std_series = values.std(axis=1, ddof=1)  # deviazione standard campionaria

lower = mean_series - std_series
upper = mean_series + std_series

# linea della media (tratteggiata e grigia)
ax.plot(
    x,
    mean_series,
    marker = 'o',
    color='grey',
    linewidth=3.0,
    label='avg_all_metrics'
)


ax.set_xlabel('Minimum weight per edge in Twitter layer')
ax.set_ylabel('Similarity metrics ')
ax.set_ylim(0.0, 1.0)

# legenda unica
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='lower right')

ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- prepara i dati (come prima) ---
metric_cols = [c for c in results_df.columns if c != 'min_weight']
X = results_df[metric_cols].values            # shape: (n_thresholds, n_metrics)
weights = results_df['min_weight'].values

# --- standardizzazione delle metriche ---
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# --- PCA ---
# puoi limitarti a n_components=2 o 3 se vuoi, qui uso tutte per vedere la varianza spiegata
pca = PCA(n_components=len(metric_cols))
X_pca = pca.fit_transform(X_std)

# PC1 è X_pca[:, 0], PC2 è X_pca[:, 1], ecc.
expl_var = pca.explained_variance_ratio_

# --- PLOT: PC1 vs soglia + varianza spiegata ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1) PC1 come funzione di min_weight
ax1 = axes[0]
ax1.plot(
    weights,
    X_pca[:, 0],
    marker='o',
    linestyle='-'
)
ax1.set_xlabel('Minimum weight per edge in Twitter layer')
ax1.set_ylabel('PC1 score')
ax1.set_title('First principal component across thresholds')
ax1.grid(True)

# 2) Varianza spiegata dalle prime componenti
ax2 = axes[1]
components = np.arange(1, len(metric_cols) + 1)
ax2.bar(components, expl_var, alpha=0.7)
ax2.set_xticks(components)
ax2.set_xlabel('Principal component')
ax2.set_ylabel('Explained variance ratio')
ax2.set_title('Variance explained by each PC')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

# Se vuoi stampare anche quanta varianza cattura PC1:
print(f"Explained variance by PC1: {expl_var[0]:.3f}")
print(f"Cumulative variance by first 2 PCs: {expl_var[:2].sum():.3f}")

In [ ]:
# plot results_df and the sum of all metric curves
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# original individual curves on ax1
for col in results_df.columns:
    if col != 'min_weight':
        ax1.plot(results_df['min_weight'],
                 results_df[col],
                 marker='o',
                 label=col,
                 alpha=0.8)



metric_cols = [c for c in results_df.columns if c != 'min_weight']
sum_series = results_df[metric_cols].sum(axis=1)



ax1.set_xlabel('Minimum Weight (Twitter edges)')
ax1.set_ylabel('Individual Similarity Metrics')

lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1 , labels1 , loc='upper right')

# grid aligned with left (ax1) y-axis
ax1.grid(True)

plt.show()


In [ ]:
# plot results_df and the sum of all metric curves
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# original individual curves on ax1
for col in results_df.columns:
    if col != 'min_weight':
        ax1.plot(results_df['min_weight'],
                 results_df[col],
                 marker='o',
                 label=col,
                 alpha=0.8)

# second y-axis for avg of all metrics
ax2 = ax1.twinx()

metric_cols = [c for c in results_df.columns if c != 'min_weight']
sum_series = results_df[metric_cols].sum(axis=1)

ax2.plot(results_df['min_weight'],
         sum_series / len(metric_cols),
         marker='o',
         color='black',
         linewidth=2.0,
         label='avg_all_metrics')

ax2.set_ylim(0., .8)

ax1.set_xlabel('Minimum Weight (Twitter edges)')
ax1.set_ylabel('Individual Similarity Metrics')
ax2.set_ylabel('Average of All Metrics')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

# grid aligned with left (ax1) y-axis
ax1.grid(True)

plt.show()


In [ ]:
# plot results_df and the sum of all metric curves
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# original individual curves on ax1
for col in results_df.columns:
    if col != 'min_weight':
        ax1.plot(results_df['min_weight'],
                 results_df[col],
                 marker='o',
                 label=col,
                 alpha=0.8)

# second y-axis for avg of all metrics
ax2 = ax1.twinx()

metric_cols = [c for c in results_df.columns if c != 'min_weight']
sum_series = results_df[metric_cols].sum(axis=1)

ax2.plot(results_df['min_weight'],
         sum_series / len(metric_cols),
         marker='o',
         color='black',
         linewidth=2.0,
         label='avg_all_metrics')

ax2.set_ylim(0., .8)

ax1.set_xlabel('Minimum Weight (Twitter edges)')
ax1.set_ylabel('Individual Similarity Metrics')
ax2.set_ylabel('Average of All Metrics')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

# grid aligned with left (ax1) y-axis
ax1.grid(True)

plt.show()


# THE WINNER IS

In [ ]:
min_weight_per_cell = 4

ress = 0.7

G_h_p = prune_by_in_degree(G_h, min_in=0)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=min_weight_per_cell, return_weighted=False)
common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())

# Build induced subgraphs containing only common nodes
G_h_common = G_h_p.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()



per_layer_df, cross_df = compare_layers(G_h_common, G_tw_b_common, use_common_nodes_only=True)
score, breakdown = similarity_index_from_compare(per_layer_df, cross_df)
print("Similarity score:", score)
display(per_layer_df)
#display(cross_df)
#display(breakdown.sort_values("contribution", ascending=False))
# Directed Louvain run for winner subgraphs (provides globals for plots/analysis)
import numpy as np
import networkx as nx

required_graphs = ['G_h_common', 'G_tw_b_common']
missing = [name for name in required_graphs if name not in globals()]
if missing:
    raise RuntimeError(f"Missing winner subgraphs: {missing}. Run the previous cell first.")

seed_registry = globals().get('winner_random_seeds')
if not isinstance(seed_registry, dict):
    seed_registry = {}
lock_seeds = bool(seed_registry.get('lock', False))
if lock_seeds:
    print("Using seeds stored in winner_random_seeds (lock=True). Set lock=False to regenerate.")
else:
    # Persist explicitly that seeds are currently unlocked/randomized
    seed_registry['lock'] = False
globals()['winner_random_seeds'] = seed_registry

rand_max = 2**31 - 1

def _ensure_seed(key, force_new=False):
    if force_new or key not in seed_registry:
        seed_registry[key] = int(np.random.randint(0, rand_max))
    return int(seed_registry[key])

force_new = not lock_seeds
layout_seed = _ensure_seed('layout', force_new=force_new)
base_seed = _ensure_seed('stability_base', force_new=force_new)
seed_h = _ensure_seed('louvain_h', force_new=force_new)
seed_tw = _ensure_seed('louvain_tw', force_new=force_new)

G_h_dir = G_h_common.copy()
G_tw_dir = G_tw_b_common.copy()
G_hu = G_h_dir.to_undirected()
G_twu = G_tw_dir.to_undirected()

com_h, lab_h, q_h_dir = _run_directed_louvain(G_h_dir, weight_attr='weight', directed=True, seed=seed_h, resolution=ress)
com_tw, lab_tw, q_tw_dir = _run_directed_louvain(G_tw_dir, weight_attr='weight', directed=True, seed=seed_tw, resolution=ress)

#print(f"Directed Louvain seeds -> Hyperlink: {seed_h}, Twitter: {seed_tw}, layout: {layout_seed}")
print(f"Communities -> Hyperlink: {len(com_h)} clusters, Twitter: {len(com_tw)} clusters")
#print(f"Directed modularity Q -> Hyperlink: {q_h_dir:.4f} (nan if unavailable)")
#print(f"Directed modularity Q -> Twitter:   {q_tw_dir:.4f} (nan if unavailable)")

cross_series = _as_cross_series(cross_df)
metrics = {
    'similarity_score': float(score) if isinstance(score, (int, float, np.floating)) else score,
    'directed_louvain_Q_h': q_h_dir,
    'directed_louvain_Q_tw': q_tw_dir,
    'n_communities_h': len(com_h),
    'n_communities_tw': len(com_tw),
}
metrics.update({f'cross_{k}': float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in cross_series.items()})
#print("Seed registry updated and metrics dict prepared for export.")
#print("Tip: set winner_random_seeds = {'lock': True, 'louvain_h': seed_h, 'louvain_tw': seed_tw, 'layout': layout_seed, 'stability_base': base_seed} to reuse them.")
# Winner graphs colored by Directed Louvain communities (shared spring layout)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx

assert 'G_h_dir' in globals() and 'G_tw_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_h' in globals() and 'lab_tw' in globals(), "Community labels not available."

seed_registry = globals().get('winner_random_seeds', {})
layout_seed = int(seed_registry.get('layout', np.random.randint(0, 2**31 - 1)))
print(f"Shared layout spring_layout seed: {layout_seed}")

def _union_layout_graph(G1, G2):
    U = nx.Graph()
    U.add_nodes_from(G1.nodes(data=True))
    for G in (G1, G2):
        for u, v, data in G.edges(data=True):
            if u == v:
                continue
            w = float(data.get('weight', 1.0))
            if U.has_edge(u, v):
                U[u][v]['weight'] = U[u][v].get('weight', 0.0) + w
            else:
                U.add_edge(u, v, weight=w)
    return U

layout_graph = _union_layout_graph( G_tw_dir, G_h_dir)
pos_shared = nx.spring_layout(layout_graph, weight='weight', seed=layout_seed)
globals()['pos_shared'] = pos_shared

_palette = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b',
    '#e377c2','#7f7f7f','#bcbd22','#17becf','#393b79','#637939',
    '#8c6d31','#843c39','#7b4173','#3182bd','#e6550d','#31a354',
    '#756bb1','#636363'
]

def _color_map_from_labels(lab):
    uniq = sorted(set(lab.values()))
    return {c: _palette[i % len(_palette)] for i, c in enumerate(uniq)}

def _style_nodes(G, base_colors, seed_size=500, disc_size=350, seed_alpha=0.9, disc_alpha=0.4):
    statuses = nx.get_node_attributes(G, 'status')
    node_sizes, styled_colors, id_labels = [], [], {}
    for node, color in zip(G.nodes(), base_colors):
        status = str(statuses.get(node, '')).lower()
        is_seed = status.startswith('seed')
        is_disc = status.startswith('disc')
        size = seed_size if is_seed else disc_size
        alpha = seed_alpha if is_seed else disc_alpha
        node_sizes.append(size)
        styled_colors.append(mcolors.to_rgba(color, alpha=alpha))
        if is_seed or is_disc:  # label both Seed and Disc nodes with their ID
            id_labels[node] = str(node)
    return node_sizes, styled_colors, id_labels

cmap_h = _color_map_from_labels(lab_h)
cmap_tw = _color_map_from_labels(lab_tw)
edge_kwargs = dict(width=0.4, alpha=0.25, arrows=False)

fig_hc, ax_hc = plt.subplots(figsize=(12, 12))
ax_hc.set_title('Hyperlink winners – Directed Louvain communities')
nx.draw_networkx_edges(G_h_dir, pos_shared, ax=ax_hc, **edge_kwargs)
node_colors_h = [cmap_h.get(lab_h.get(n, -1), '#cccccc') for n in G_h_dir.nodes()]
node_sizes_h, styled_colors_h, seed_labels_h = _style_nodes(G_h_dir, node_colors_h)
nx.draw_networkx_nodes(G_h_dir, pos_shared, ax=ax_hc,
                       node_color=styled_colors_h,
                       node_size=node_sizes_h,
                       edgecolors='black', linewidths=0.6)
if seed_labels_h:
    nx.draw_networkx_labels(G_h_dir, pos_shared, labels=seed_labels_h,
                            font_size=9, font_color='black', ax=ax_hc)
ax_hc.axis('off')
handles_h = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                        markerfacecolor=cmap_h[c], markeredgecolor='black',
                        label=f'Comm {c}') for c in sorted(cmap_h)]
if handles_h:
    ax_hc.legend(handles=handles_h, frameon=False, loc='upper left', ncol=2)
plt.show()

fig_tc, ax_tc = plt.subplots(figsize=(12, 12))
ax_tc.set_title('Twitter winners – Directed Louvain communities')
nx.draw_networkx_edges(G_tw_dir, pos_shared, ax=ax_tc, **edge_kwargs)
node_colors_tw = [cmap_tw.get(lab_tw.get(n, -1), '#cccccc') for n in G_tw_dir.nodes()]
node_sizes_tw, styled_colors_tw, seed_labels_tw = _style_nodes(G_tw_dir, node_colors_tw)
nx.draw_networkx_nodes(G_tw_dir, pos_shared, ax=ax_tc,
                       node_color=styled_colors_tw,
                       node_size=node_sizes_tw,
                       edgecolors='black', linewidths=0.6)
if seed_labels_tw:
    nx.draw_networkx_labels(G_tw_dir, pos_shared, labels=seed_labels_tw,
                            font_size=9, font_color='black', ax=ax_tc)
ax_tc.axis('off')
handles_tw = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                         markerfacecolor=cmap_tw[c], markeredgecolor='black',
                         label=f'Comm {c}') for c in sorted(cmap_tw)]
if handles_tw:
    ax_tc.legend(handles=handles_tw, frameon=False, loc='upper left', ncol=2)
plt.show()

print("Community-colored visualizations generated with shared spring layout (IDs for Seed and Disc nodes).")

In [ ]:
min_weight_per_cell = 14

ress = 0.27

G_h_p = prune_by_in_degree(G_h, min_in=0)
G_tw_b = build_twitter_graphs(tw_nodes, weighted_tw_edges, min_weight=min_weight_per_cell, return_weighted=False)
common_nodes = set(G_h_p.nodes()) & set(G_tw_b.nodes())

# Build induced subgraphs containing only common nodes
G_h_common = G_h_p.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b.subgraph(common_nodes).copy()

# take the LCC
G_h_common = G_h_common.subgraph(max(nx.weakly_connected_components(G_h_common), key=len)).copy()
G_tw_b_common = G_tw_b_common.subgraph(max(nx.weakly_connected_components(G_tw_b_common), key=len)).copy()

common_nodes = set(G_h_common.nodes()) & set(G_tw_b_common.nodes())
# Build induced subgraphs containing only common nodes
G_h_common = G_h_common.subgraph(common_nodes).copy()
G_tw_b_common = G_tw_b_common.subgraph(common_nodes).copy()



per_layer_df, cross_df = compare_layers(G_h_common, G_tw_b_common, use_common_nodes_only=True)
score, breakdown = similarity_index_from_compare(per_layer_df, cross_df)
print("Similarity score:", score)
display(per_layer_df)
#display(cross_df)
#display(breakdown.sort_values("contribution", ascending=False))
# Directed Louvain run for winner subgraphs (provides globals for plots/analysis)
import numpy as np
import networkx as nx

required_graphs = ['G_h_common', 'G_tw_b_common']
missing = [name for name in required_graphs if name not in globals()]
if missing:
    raise RuntimeError(f"Missing winner subgraphs: {missing}. Run the previous cell first.")

seed_registry = globals().get('winner_random_seeds')
if not isinstance(seed_registry, dict):
    seed_registry = {}
lock_seeds = bool(seed_registry.get('lock', False))
if lock_seeds:
    print("Using seeds stored in winner_random_seeds (lock=True). Set lock=False to regenerate.")
else:
    # Persist explicitly that seeds are currently unlocked/randomized
    seed_registry['lock'] = False
globals()['winner_random_seeds'] = seed_registry

rand_max = 2**31 - 1

def _ensure_seed(key, force_new=False):
    if force_new or key not in seed_registry:
        seed_registry[key] = int(np.random.randint(0, rand_max))
    return int(seed_registry[key])

force_new = not lock_seeds
layout_seed = _ensure_seed('layout', force_new=force_new)
base_seed = _ensure_seed('stability_base', force_new=force_new)
seed_h = _ensure_seed('louvain_h', force_new=force_new)
seed_tw = _ensure_seed('louvain_tw', force_new=force_new)

G_h_dir = G_h_common.copy()
G_tw_dir = G_tw_b_common.copy()
G_hu = G_h_dir.to_undirected()
G_twu = G_tw_dir.to_undirected()

com_h, lab_h, q_h_dir = _run_directed_louvain(G_h_dir, weight_attr='weight', directed=True, seed=seed_h, resolution=ress)
com_tw, lab_tw, q_tw_dir = _run_directed_louvain(G_tw_dir, weight_attr='weight', directed=True, seed=seed_tw, resolution=ress)

#print(f"Directed Louvain seeds -> Hyperlink: {seed_h}, Twitter: {seed_tw}, layout: {layout_seed}")
print(f"Communities -> Hyperlink: {len(com_h)} clusters, Twitter: {len(com_tw)} clusters")
#print(f"Directed modularity Q -> Hyperlink: {q_h_dir:.4f} (nan if unavailable)")
#print(f"Directed modularity Q -> Twitter:   {q_tw_dir:.4f} (nan if unavailable)")

cross_series = _as_cross_series(cross_df)
metrics = {
    'similarity_score': float(score) if isinstance(score, (int, float, np.floating)) else score,
    'directed_louvain_Q_h': q_h_dir,
    'directed_louvain_Q_tw': q_tw_dir,
    'n_communities_h': len(com_h),
    'n_communities_tw': len(com_tw),
}
metrics.update({f'cross_{k}': float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in cross_series.items()})
#print("Seed registry updated and metrics dict prepared for export.")
#print("Tip: set winner_random_seeds = {'lock': True, 'louvain_h': seed_h, 'louvain_tw': seed_tw, 'layout': layout_seed, 'stability_base': base_seed} to reuse them.")
# Winner graphs colored by Directed Louvain communities (shared spring layout)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx

assert 'G_h_dir' in globals() and 'G_tw_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_h' in globals() and 'lab_tw' in globals(), "Community labels not available."

seed_registry = globals().get('winner_random_seeds', {})
layout_seed = int(seed_registry.get('layout', np.random.randint(0, 2**31 - 1)))
print(f"Shared layout spring_layout seed: {layout_seed}")

def _union_layout_graph(G1, G2):
    U = nx.Graph()
    U.add_nodes_from(G1.nodes(data=True))
    for G in (G1, G2):
        for u, v, data in G.edges(data=True):
            if u == v:
                continue
            w = float(data.get('weight', 1.0))
            if U.has_edge(u, v):
                U[u][v]['weight'] = U[u][v].get('weight', 0.0) + w
            else:
                U.add_edge(u, v, weight=w)
    return U

layout_graph = _union_layout_graph( G_tw_dir, G_h_dir)
pos_shared = nx.spring_layout(layout_graph, weight='weight', seed=layout_seed)
globals()['pos_shared'] = pos_shared

_palette = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b',
    '#e377c2','#7f7f7f','#bcbd22','#17becf','#393b79','#637939',
    '#8c6d31','#843c39','#7b4173','#3182bd','#e6550d','#31a354',
    '#756bb1','#636363'
]

def _color_map_from_labels(lab):
    uniq = sorted(set(lab.values()))
    return {c: _palette[i % len(_palette)] for i, c in enumerate(uniq)}

def _style_nodes(G, base_colors, seed_size=500, disc_size=350, seed_alpha=0.9, disc_alpha=0.4):
    statuses = nx.get_node_attributes(G, 'status')
    node_sizes, styled_colors, id_labels = [], [], {}
    for node, color in zip(G.nodes(), base_colors):
        status = str(statuses.get(node, '')).lower()
        is_seed = status.startswith('seed')
        is_disc = status.startswith('disc')
        size = seed_size if is_seed else disc_size
        alpha = seed_alpha if is_seed else disc_alpha
        node_sizes.append(size)
        styled_colors.append(mcolors.to_rgba(color, alpha=alpha))
        if is_seed or is_disc:  # label both Seed and Disc nodes with their ID
            id_labels[node] = str(node)
    return node_sizes, styled_colors, id_labels

cmap_h = _color_map_from_labels(lab_h)
cmap_tw = _color_map_from_labels(lab_tw)
edge_kwargs = dict(width=0.4, alpha=0.25, arrows=False)

fig_hc, ax_hc = plt.subplots(figsize=(12, 12))
ax_hc.set_title('Hyperlink winners – Directed Louvain communities')
nx.draw_networkx_edges(G_h_dir, pos_shared, ax=ax_hc, **edge_kwargs)
node_colors_h = [cmap_h.get(lab_h.get(n, -1), '#cccccc') for n in G_h_dir.nodes()]
node_sizes_h, styled_colors_h, seed_labels_h = _style_nodes(G_h_dir, node_colors_h)
nx.draw_networkx_nodes(G_h_dir, pos_shared, ax=ax_hc,
                       node_color=styled_colors_h,
                       node_size=node_sizes_h,
                       edgecolors='black', linewidths=0.6)
if seed_labels_h:
    nx.draw_networkx_labels(G_h_dir, pos_shared, labels=seed_labels_h,
                            font_size=9, font_color='black', ax=ax_hc)
ax_hc.axis('off')
handles_h = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                        markerfacecolor=cmap_h[c], markeredgecolor='black',
                        label=f'Comm {c}') for c in sorted(cmap_h)]
if handles_h:
    ax_hc.legend(handles=handles_h, frameon=False, loc='upper left', ncol=2)
plt.show()

fig_tc, ax_tc = plt.subplots(figsize=(12, 12))
ax_tc.set_title('Twitter winners – Directed Louvain communities')
nx.draw_networkx_edges(G_tw_dir, pos_shared, ax=ax_tc, **edge_kwargs)
node_colors_tw = [cmap_tw.get(lab_tw.get(n, -1), '#cccccc') for n in G_tw_dir.nodes()]
node_sizes_tw, styled_colors_tw, seed_labels_tw = _style_nodes(G_tw_dir, node_colors_tw)
nx.draw_networkx_nodes(G_tw_dir, pos_shared, ax=ax_tc,
                       node_color=styled_colors_tw,
                       node_size=node_sizes_tw,
                       edgecolors='black', linewidths=0.6)
if seed_labels_tw:
    nx.draw_networkx_labels(G_tw_dir, pos_shared, labels=seed_labels_tw,
                            font_size=9, font_color='black', ax=ax_tc)
ax_tc.axis('off')
handles_tw = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                         markerfacecolor=cmap_tw[c], markeredgecolor='black',
                         label=f'Comm {c}') for c in sorted(cmap_tw)]
if handles_tw:
    ax_tc.legend(handles=handles_tw, frameon=False, loc='upper left', ncol=2)
plt.show()

print("Community-colored visualizations generated with shared spring layout (IDs for Seed and Disc nodes).")

In [ ]:
# Global properties for the common-node subgraphs (Hyperlink vs Twitter)
import numpy as np
import pandas as pd
import networkx as nx

assert 'G_h_common' in globals() and 'G_tw_b_common' in globals(), "Run the preprocessing cells to build the common subgraphs first."

def _all_pairs_path_stats(UG):
    lengths = []
    diameters = []
    for component_nodes in nx.connected_components(UG):
        if len(component_nodes) < 2:
            continue
        sub = UG.subgraph(component_nodes).copy()
        sp = dict(nx.all_pairs_shortest_path_length(sub))
        ordered_nodes = list(sub.nodes())
        for i, src in enumerate(ordered_nodes):
            for dst in ordered_nodes[i+1:]:
                lengths.append(sp[src][dst])
        try:
            diameters.append(nx.diameter(sub))
        except Exception:
            pass
    if not lengths:
        return np.nan, np.nan, np.nan
    return (float(np.mean(lengths)), float(np.std(lengths, ddof=0)), int(max(diameters)) if diameters else np.nan)

def _summarize_common_graph(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = _directed_density(G)
    deg_vals = np.array([G.in_degree(v) + G.out_degree(v) for v in G.nodes()], dtype=float)
    strength_vals = np.array([G.degree(v, weight='weight') for v in G.nodes()], dtype=float)

    def _stat(vec):
        if vec.size == 0:
            return (np.nan, np.nan)
        return float(vec.mean()), float(vec.std(ddof=0))

    deg_mean, deg_std = _stat(deg_vals)
    strength_mean, strength_std = _stat(strength_vals)

    UG = _UG(G)
    assort = nx.degree_assortativity_coefficient(UG) if UG.number_of_edges() else np.nan
    clust_vals = np.array(list(nx.clustering(UG, weight='weight').values()), dtype=float) if UG.number_of_nodes() else np.array([])
    clust_mean, clust_std = _stat(clust_vals)

    path_mean, path_std, diameter = _all_pairs_path_stats(UG)
    return dict(
        nodes=int(n),
        edges=int(m),
        density=float(density),
        avg_degree_mean=deg_mean,
        avg_degree_std=deg_std,
        avg_strength_mean=strength_mean,
        avg_strength_std=strength_std,
        assortativity=float(assort) if not np.isnan(assort) else np.nan,
        clust_mean=clust_mean,
        clust_std=clust_std,
        path_mean=path_mean,
        path_std=path_std,
        diameter=int(diameter) if isinstance(diameter, (int, np.integer)) else diameter
    )

stats_common = {
    'Hyperlink': _summarize_common_graph(G_h_common),
    'Twitter': _summarize_common_graph(G_tw_b_common)
}

def _fmt_mean_std(mean_val, std_val, digits=3):
    if np.isnan(mean_val):
        return "--"
    if np.isnan(std_val):
        return f"{mean_val:.{digits}f}"
    return f"{mean_val:.{digits}f} ({std_val:.{digits}f})"

def _fmt_scalar(val, digits=3):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "--"
    fmt = f"{{:.{digits}f}}"
    if isinstance(val, (int, np.integer)):
        return str(int(val))
    return fmt.format(val)

rows = [
    ("Nr. nodes", {layer: stats_common[layer]['nodes'] for layer in stats_common}),
    ("Nr. edges", {layer: stats_common[layer]['edges'] for layer in stats_common}),
    ("Density", {layer: _fmt_scalar(stats_common[layer]['density']) for layer in stats_common}),
    ("Avg.degree (std.dev.)", {layer: _fmt_mean_std(stats_common[layer]['avg_degree_mean'], stats_common[layer]['avg_degree_std']) for layer in stats_common}),
    ("Avg.strength (std.dev.)", {layer: _fmt_mean_std(stats_common[layer]['avg_strength_mean'], stats_common[layer]['avg_strength_std']) for layer in stats_common}),
    ("Assortativity", {layer: _fmt_scalar(stats_common[layer]['assortativity'], digits=4) for layer in stats_common}),
    ("Avg.clust.coeff. (std.dev)", {layer: _fmt_mean_std(stats_common[layer]['clust_mean'], stats_common[layer]['clust_std']) for layer in stats_common}),
    ("Avg.path length (std.dev)", {layer: _fmt_mean_std(stats_common[layer]['path_mean'], stats_common[layer]['path_std']) for layer in stats_common}),
    ("Diameter", {layer: _fmt_scalar(stats_common[layer]['diameter'], digits=0) for layer in stats_common})
 ]

table_common = pd.DataFrame({metric: values for metric, values in rows}).T
table_common.columns = ['Hyperlink', 'Twitter']
display(table_common)
print(table_common.to_latex(escape=False, caption='Global properties of the networks restricted to common nodes', label='tab:global_prop_common'))

In [ ]:
# Sankey diagram mapping Hyperlink communities to Twitter communities
import plotly.graph_objects as go
from IPython.display import display, HTML
from collections import Counter
import matplotlib.colors as mcolors

assert 'lab_h' in globals() and 'lab_tw' in globals(), "Run the Directed Louvain cell first."
if 'common_nodes' in globals():
    common_pool = sorted(common_nodes)
else:
    common_pool = sorted(set(lab_h.keys()).intersection(lab_tw.keys()))

flows = Counter()
for node in common_pool:
    comm_h = lab_h.get(node)
    comm_tw = lab_tw.get(node)
    if comm_h is None or comm_tw is None:
        continue
    flows[(comm_h, comm_tw)] += 1
if not flows:
    raise ValueError("No overlapping nodes with community labels to plot.")

hyper_comms = sorted({h for h, _ in flows})
twitter_comms = sorted({t for _, t in flows})
count_h = {c: 0 for c in hyper_comms}
count_tw = {c: 0 for c in twitter_comms}
for (comm_h, comm_tw), value in flows.items():
    count_h[comm_h] += value
    count_tw[comm_tw] += value

_palette = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
    '#393b79','#637939','#8c6d31','#843c39','#7b4173','#3182bd','#e6550d','#31a354','#756bb1','#636363'
]

def _ensure_color_map(comm_list, existing):
    cmap = {}
    for idx, comm in enumerate(comm_list):
        if isinstance(existing, dict) and comm in existing:
            cmap[comm] = existing[comm]
        else:
            cmap[comm] = _palette[idx % len(_palette)]
    return cmap

cmap_h_local = _ensure_color_map(hyper_comms, globals().get('cmap_h'))
cmap_tw_local = _ensure_color_map(twitter_comms, globals().get('cmap_tw'))

node_labels = [f"Hyperlink c{cid} (n={count_h[cid]})" for cid in hyper_comms]
node_labels += [f"Twitter c{cid} (n={count_tw[cid]})" for cid in twitter_comms]
node_colors = [cmap_h_local[cid] for cid in hyper_comms] + [cmap_tw_local[cid] for cid in twitter_comms]

hyper_index = {cid: idx for idx, cid in enumerate(hyper_comms)}
twitter_index = {cid: len(hyper_comms) + idx for idx, cid in enumerate(twitter_comms)}

links_sorted = sorted(flows.items(), key=lambda kv: (kv[0][0], kv[0][1]))
sources, targets, values, link_labels, link_colors = [], [], [], [], []
for (comm_h, comm_tw), value in links_sorted:
    sources.append(hyper_index[comm_h])
    targets.append(twitter_index[comm_tw])
    values.append(value)
    link_labels.append(f"{value} nodes: H{comm_h} -> T{comm_tw}")
    rgba = mcolors.to_rgba(cmap_h_local.get(comm_h, '#999999'), alpha=0.55)
    link_colors.append(f"rgba({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)}, {rgba[3]:.2f})")

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=node_labels,
        color=node_colors,
        pad=22,
        thickness=18
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        label=link_labels,
        color=link_colors
    )
))
fig.update_layout(
    title="Community membership flow: Hyperlink -> Twitter",
    font=dict(size=12)
 )
display(HTML(fig.to_html(include_plotlyjs='cdn')))
print(f"Sankey includes {len(values)} flows across {len(hyper_comms)} Hyperlink and {len(twitter_comms)} Twitter communities.")

In [ ]:
from collections import Counter

# 4 pie charts: category distribution inside the two detected communities
# for both Hyperlink (G_h_dir, lab_h) and Twitter (G_tw_dir, lab_tw)

assert 'G_h_dir' in globals() and 'G_tw_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_h' in globals() and 'lab_tw' in globals(), "Community labels not found."
assert 'color_map_cat' in globals(), "Category color map (color_map_cat) not defined."

# pick the two target communities (use those computed below if available; otherwise top-2 by size)

if 'comm_ids' in globals() and isinstance(comm_ids, (list, tuple)) and len(comm_ids) == 2:
    hyper_ids = list(comm_ids)
else:
    hyper_ids = [c for c, _ in Counter(lab_h.values()).most_common(2)]

if 'tw_comm_ids' in globals() and isinstance(tw_comm_ids, (list, tuple)) and len(tw_comm_ids) == 2:
    twitter_ids = list(tw_comm_ids)
else:
    twitter_ids = [c for c, _ in Counter(lab_tw.values()).most_common(2)]

def community_category_counts(G, labels, comm_id):
    members = [n for n, c in labels.items() if c == comm_id and n in G]
    cats = [G.nodes[n].get('category', 'Unknown') for n in members]
    return Counter(cats), len(members)

def make_pie(ax, counts, title):
    if not counts or sum(counts.values()) == 0:
        ax.text(0.5, 0.5, "No nodes", ha='center', va='center')
        ax.set_title(title)
        ax.axis('off')
        return
    labels = list(counts.keys())
    sizes = [counts[k] for k in labels]
    colors = [color_map_cat.get(cat, '#cccccc') for cat in labels]
    def _autopct_fmt(vals):
        total = sum(vals)
        def _f(pct):
            cnt = int(round(pct * total / 100.0))
            return f"{pct:.1f}%\n(n={cnt})" if cnt > 0 else ""
        return _f
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors, startangle=90, counterclock=False,
        autopct=_autopct_fmt(sizes), pctdistance=0.75, textprops=dict(color='black', fontsize=11),
        wedgeprops=dict(alpha=0.7)  # set slice alpha
    )
    ax.set_title(title)
    ax.axis('equal')

# compute counts
h_cnt_0, h_n0 = community_category_counts(G_h_dir, lab_h, hyper_ids[0])
h_cnt_1, h_n1 = community_category_counts(G_h_dir, lab_h, hyper_ids[1])
t_cnt_0, t_n0 = community_category_counts(G_tw_dir, lab_tw, twitter_ids[0])
t_cnt_1, t_n1 = community_category_counts(G_tw_dir, lab_tw, twitter_ids[1])

# plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
make_pie(axes[0, 0], h_cnt_0, f"Hyperlink – Community {hyper_ids[0]} (n={h_n0})")
make_pie(axes[0, 1], h_cnt_1, f"Hyperlink – Community {hyper_ids[1]} (n={h_n1})")
make_pie(axes[1, 0], t_cnt_0, f"Twitter – Community {twitter_ids[0]} (n={t_n0})")
make_pie(axes[1, 1], t_cnt_1, f"Twitter – Community {twitter_ids[1]} (n={t_n1})")
fig.suptitle("Category distribution inside the two detected communities (Hyperlink vs Twitter)", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

# Discovered nodes that bridge between the two Hyperlink communities (winners)

# prerequisites
assert 'G_hu' in globals() and 'G_h_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_h' in globals(), "Hyperlink community labels (lab_h) not found."


# use the two largest communities if more than 2 exist
comm_ids = list(sorted(set(lab_h.values())))
if len(comm_ids) != 2:
    sizes = Counter(lab_h.values())
    top2 = [c for c, _ in sizes.most_common(2)]
    comm_ids = sorted(top2)

comm_set = set(comm_ids)
print(f"Analyzing bridges between Hyperlink communities: {comm_ids}")

def _status_is_disc(attrs):
    return str(attrs.get('status', '')).lower().startswith('disc')

rows = []
for n, d in G_hu.nodes(data=True):
    if not _status_is_disc(d):
        continue
    own = lab_h.get(n, None)
    if own is None:
        continue
    # restrict to the two selected communities
    if own not in comm_set:
        continue

    neigh = set(G_hu.neighbors(n))
    neigh_comm = [lab_h.get(v, None) for v in neigh if lab_h.get(v, None) in comm_set]
    neigh_comm_set = set(neigh_comm)

    # cross-community neighbors (undirected view)
    cross_neigh = [v for v in neigh if lab_h.get(v, None) in comm_set and lab_h.get(v) != own]
    # cross-community incident edges (directed in+out)
    cross_out = [v for v in G_h_dir.successors(n) if lab_h.get(v, None) in comm_set and lab_h.get(v) != own]
    cross_in  = [u for u in G_h_dir.predecessors(n) if lab_h.get(u, None) in comm_set and lab_h.get(u) != own]
    cross_edges = len(cross_out) + len(cross_in)

    deg_u = G_hu.degree(n)
    cross_frac = (len(cross_neigh) / deg_u) if deg_u > 0 else 0.0

    rows.append(dict(
        node=n,
        label=G_h_dir.nodes[n].get('label'),
        own_comm=int(own),
        deg=deg_u,
        cross_neighbors=len(cross_neigh),
        cross_edges=cross_edges,
        cross_fraction=cross_frac,
        neighbor_comms=sorted(neigh_comm_set),
        is_bridge_strict=(neigh_comm_set == comm_set)  # touches both communities
    ))

bridge_df_h = pd.DataFrame(rows)
if bridge_df_h.empty:
    print("No discovered nodes found.")
else:
    # strict bridges: discovered nodes with neighbors in both communities
    bridge_nodes_disc_h = (
        bridge_df_h[bridge_df_h['is_bridge_strict']]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    relaxed_connectors_h = (
        bridge_df_h[bridge_df_h['cross_neighbors'] > 0]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    print("Discovered nodes that behave like a bridge between the two Hyperlink communities (strict):")
    display(bridge_nodes_disc_h[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

    print("Discovered nodes connecting across communities (relaxed, at least one cross-community neighbor):")
    display(relaxed_connectors_h[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

In [ ]:
from collections import Counter

# Discovered nodes that bridge between the two Hyperlink communities (winners)

# prerequisites
assert 'G_hu' in globals() and 'G_h_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_h' in globals(), "Hyperlink community labels (lab_h) not found."


# use the two largest communities if more than 2 exist
comm_ids = list(sorted(set(lab_h.values())))
if len(comm_ids) != 2:
    sizes = Counter(lab_h.values())
    top2 = [c for c, _ in sizes.most_common(2)]
    comm_ids = sorted(top2)

comm_set = set(comm_ids)
print(f"Analyzing bridges between Hyperlink communities: {comm_ids}")

def _status_is_disc(attrs):
    return True
    #return str(attrs.get('status', '')).lower().startswith('disc')

rows = []
for n, d in G_hu.nodes(data=True):
    if not _status_is_disc(d):
        continue
    own = lab_h.get(n, None)
    if own is None:
        continue
    # restrict to the two selected communities
    if own not in comm_set:
        continue

    neigh = set(G_hu.neighbors(n))
    neigh_comm = [lab_h.get(v, None) for v in neigh if lab_h.get(v, None) in comm_set]
    neigh_comm_set = set(neigh_comm)

    # cross-community neighbors (undirected view)
    cross_neigh = [v for v in neigh if lab_h.get(v, None) in comm_set and lab_h.get(v) != own]
    # cross-community incident edges (directed in+out)
    cross_out = [v for v in G_h_dir.successors(n) if lab_h.get(v, None) in comm_set and lab_h.get(v) != own]
    cross_in  = [u for u in G_h_dir.predecessors(n) if lab_h.get(u, None) in comm_set and lab_h.get(u) != own]
    cross_edges = len(cross_out) + len(cross_in)

    deg_u = G_hu.degree(n)
    cross_frac = (len(cross_neigh) / deg_u) if deg_u > 0 else 0.0

    rows.append(dict(
        node=n,
        label=G_h_dir.nodes[n].get('label'),
        own_comm=int(own),
        deg=deg_u,
        cross_neighbors=len(cross_neigh),
        cross_edges=cross_edges,
        cross_fraction=cross_frac,
        neighbor_comms=sorted(neigh_comm_set),
        is_bridge_strict=(neigh_comm_set == comm_set)  # touches both communities
    ))

bridge_df_h = pd.DataFrame(rows)
if bridge_df_h.empty:
    print("No discovered nodes found.")
else:
    # strict bridges: discovered nodes with neighbors in both communities
    bridge_nodes_disc_h = (
        bridge_df_h[bridge_df_h['is_bridge_strict']]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    relaxed_connectors_h = (
        bridge_df_h[bridge_df_h['cross_neighbors'] > 0]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    print("Discovered nodes that behave like a bridge between the two Hyperlink communities (strict):")
    display(bridge_nodes_disc_h[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

    print("Discovered nodes connecting across communities (relaxed, at least one cross-community neighbor):")
    display(relaxed_connectors_h[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

In [ ]:
# Discovered nodes that bridge between the two largest Twitter communities (winner subgraphs)

assert 'G_twu' in globals() and 'G_tw_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_tw' in globals(), "Twitter community labels (lab_tw) not found."

# select the two largest communities
tw_comm_ids = list(sorted(set(lab_tw.values())))
if len(tw_comm_ids) != 2:
    # take top-2 by size
    sizes_tw = Counter(lab_tw.values())
    top2_tw = [c for c, _ in sizes_tw.most_common(2)]
    tw_comm_ids = sorted(top2_tw)

tw_comm_set = set(tw_comm_ids)
print(f"Analyzing bridges between Twitter communities: {tw_comm_ids}")

def _status_is_disc(attrs):
    #return True
    return str(attrs.get('status', '')).lower().startswith('disc')

rows_tw = []
for n, d in G_twu.nodes(data=True):
    if not _status_is_disc(d):
        continue
    own = lab_tw.get(n, None)
    if own is None or own not in tw_comm_set:
        continue

    neigh = set(G_twu.neighbors(n))
    neigh_comm = [lab_tw.get(v, None) for v in neigh if lab_tw.get(v, None) in tw_comm_set]
    neigh_comm_set = set(neigh_comm)

    cross_neigh = [v for v in neigh if lab_tw.get(v, None) in tw_comm_set and lab_tw.get(v) != own]
    cross_out = [v for v in G_tw_dir.successors(n) if lab_tw.get(v, None) in tw_comm_set and lab_tw.get(v) != own]
    cross_in  = [u for u in G_tw_dir.predecessors(n) if lab_tw.get(u, None) in tw_comm_set and lab_tw.get(u) != own]
    cross_edges = len(cross_out) + len(cross_in)

    deg_u = G_twu.degree(n)
    cross_frac = (len(cross_neigh) / deg_u) if deg_u > 0 else 0.0

    rows_tw.append(dict(
        node=n,
        label=G_tw_dir.nodes[n].get('label'),
        own_comm=int(own),
        deg=deg_u,
        cross_neighbors=len(cross_neigh),
        cross_edges=cross_edges,
        cross_fraction=cross_frac,
        neighbor_comms=sorted(neigh_comm_set),
        is_bridge_strict=(neigh_comm_set == tw_comm_set)
    ))

bridge_df_tw = pd.DataFrame(rows_tw)
if bridge_df_tw.empty:
    print("No discovered Twitter nodes found.")
else:
    bridge_nodes_disc_tw = (
        bridge_df_tw[bridge_df_tw['is_bridge_strict']]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    relaxed_connectors_tw = (
        bridge_df_tw[bridge_df_tw['cross_neighbors'] > 0]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    print("Discovered Twitter nodes bridging both selected communities (strict):")
    display(bridge_nodes_disc_tw[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

    print("Discovered Twitter nodes with at least one cross-community neighbor (relaxed):")
    display(relaxed_connectors_tw[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))


In [ ]:
# Discovered nodes that bridge between the two largest Twitter communities (winner subgraphs)

assert 'G_twu' in globals() and 'G_tw_dir' in globals(), "Run the Directed Louvain cell first."
assert 'lab_tw' in globals(), "Twitter community labels (lab_tw) not found."

# select the two largest communities
tw_comm_ids = list(sorted(set(lab_tw.values())))
if len(tw_comm_ids) != 2:
    # take top-2 by size
    sizes_tw = Counter(lab_tw.values())
    top2_tw = [c for c, _ in sizes_tw.most_common(2)]
    tw_comm_ids = sorted(top2_tw)

tw_comm_set = set(tw_comm_ids)
print(f"Analyzing bridges between Twitter communities: {tw_comm_ids}")

def _status_is_disc(attrs):
    return True
    #return str(attrs.get('status', '')).lower().startswith('disc')

rows_tw = []
for n, d in G_twu.nodes(data=True):
    if not _status_is_disc(d):
        continue
    own = lab_tw.get(n, None)
    if own is None or own not in tw_comm_set:
        continue

    neigh = set(G_twu.neighbors(n))
    neigh_comm = [lab_tw.get(v, None) for v in neigh if lab_tw.get(v, None) in tw_comm_set]
    neigh_comm_set = set(neigh_comm)

    cross_neigh = [v for v in neigh if lab_tw.get(v, None) in tw_comm_set and lab_tw.get(v) != own]
    cross_out = [v for v in G_tw_dir.successors(n) if lab_tw.get(v, None) in tw_comm_set and lab_tw.get(v) != own]
    cross_in  = [u for u in G_tw_dir.predecessors(n) if lab_tw.get(u, None) in tw_comm_set and lab_tw.get(u) != own]
    cross_edges = len(cross_out) + len(cross_in)

    deg_u = G_twu.degree(n)
    cross_frac = (len(cross_neigh) / deg_u) if deg_u > 0 else 0.0

    rows_tw.append(dict(
        node=n,
        label=G_tw_dir.nodes[n].get('label'),
        own_comm=int(own),
        deg=deg_u,
        cross_neighbors=len(cross_neigh),
        cross_edges=cross_edges,
        cross_fraction=cross_frac,
        neighbor_comms=sorted(neigh_comm_set),
        is_bridge_strict=(neigh_comm_set == tw_comm_set)
    ))

bridge_df_tw = pd.DataFrame(rows_tw)
if bridge_df_tw.empty:
    print("No discovered Twitter nodes found.")
else:
    bridge_nodes_disc_tw = (
        bridge_df_tw[bridge_df_tw['is_bridge_strict']]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    relaxed_connectors_tw = (
        bridge_df_tw[bridge_df_tw['cross_neighbors'] > 0]
        .sort_values(['cross_edges', 'cross_fraction', 'deg'], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    print("Discovered Twitter nodes bridging both selected communities (strict):")
    display(bridge_nodes_disc_tw[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))

    print("Discovered Twitter nodes with at least one cross-community neighbor (relaxed):")
    display(relaxed_connectors_tw[['node','label','own_comm','deg','cross_neighbors','cross_edges','cross_fraction']].head(20))


In [ ]:
# Extended community analysis: modularity, stability, category composition, exports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import networkx as nx

from collections import Counter
try:
    from networkx.algorithms.community import modularity
except Exception:
    from networkx.algorithms.community.quality import modularity

# Ensure previous community detection has run
assert 'lab_h' in globals() and 'lab_tw' in globals(), "Run the previous community cell first."
assert 'G_hu' in globals() and 'G_twu' in globals(), "Undirected graphs G_hu and G_twu not found."
assert 'G_h_dir' in globals() and 'G_tw_dir' in globals(), "Directed graphs for Directed Louvain not found."

seed_registry = globals().get('winner_random_seeds', {})
stability_base_seed = int(seed_registry.get('stability_base', np.random.randint(0, 2**31 - 1)))
print(f"Stability base seed: {stability_base_seed}")

# --------------------------------------------------
# 1. Modularity for each layer
# --------------------------------------------------
comms_h = {}
for n, c in lab_h.items():
    comms_h.setdefault(c, set()).add(n)
comms_tw = {}
for n, c in lab_tw.items():
    comms_tw.setdefault(c, set()).add(n)

Q_h = modularity(G_hu, list(comms_h.values()), weight='weight') if G_hu.number_of_edges() > 0 else np.nan
Q_tw = modularity(G_twu, list(comms_tw.values()), weight='weight') if G_twu.number_of_edges() > 0 else np.nan

print(f"Modularity (Hyperlink): {Q_h:.4f}" if not np.isnan(Q_h) else "Modularity (Hyperlink): --")
print(f"Modularity (Twitter):   {Q_tw:.4f}" if not np.isnan(Q_tw) else "Modularity (Twitter): --")

# --------------------------------------------------
# 2. Stability (multiple Directed Louvain runs -> ARI distribution)
# --------------------------------------------------
stability_stats = {
    'avg_ari_h': np.nan, 'std_ari_h': np.nan,
    'avg_ari_tw': np.nan, 'std_ari_tw': np.nan
}
if '_HAS_SK' in globals() and _HAS_SK:
    from sklearn.metrics import adjusted_rand_score

    def _multi_louvain_aris(G, directed=True, runs=10, base_seed=0):
        """Run Directed Louvain several times and return pairwise ARIs for stability."""
        nodes = list(G.nodes())
        if len(nodes) < 2:
            return np.array([])
        label_runs = []
        for s in range(runs):
            _, lab_tmp, _ = _run_directed_louvain(G, weight_attr='weight', directed=directed, seed=base_seed + s)
            label_runs.append(np.array([lab_tmp.get(n, -1) for n in nodes], dtype=int))
        aris = []
        for i in range(len(label_runs)):
            for j in range(i+1, len(label_runs)):
                aris.append(adjusted_rand_score(label_runs[i], label_runs[j]))
        return np.array(aris, dtype=float)

    aris_h = _multi_louvain_aris(G_h_dir, directed=True, runs=10, base_seed=stability_base_seed) if G_h_dir.number_of_nodes()>1 else np.array([])
    aris_tw = _multi_louvain_aris(G_tw_dir, directed=True, runs=10, base_seed=stability_base_seed + 1000) if G_tw_dir.number_of_nodes()>1 else np.array([])
    if aris_h.size:
        stability_stats['avg_ari_h'] = float(aris_h.mean())
        stability_stats['std_ari_h'] = float(aris_h.std(ddof=0))
    if aris_tw.size:
        stability_stats['avg_ari_tw'] = float(aris_tw.mean())
        stability_stats['std_ari_tw'] = float(aris_tw.std(ddof=0))
    print(f"Stability (Hyperlink) ARI mean={stability_stats['avg_ari_h']:.3f} std={stability_stats['std_ari_h']:.3f}" if aris_h.size else "Stability (Hyperlink) ARI: --")
    print(f"Stability (Twitter)   ARI mean={stability_stats['avg_ari_tw']:.3f} std={stability_stats['std_ari_tw']:.3f}" if aris_tw.size else "Stability (Twitter) ARI: --")
else:
    print("Stability test skipped (Directed Louvain or scikit-learn not available).")

# --------------------------------------------------
# 3. Category composition per community
# --------------------------------------------------
def _community_category_table(G, lab):
    cat_attr = 'category'
    rows = []
    categories = sorted({G.nodes[n].get(cat_attr, 'Unknown') for n in G.nodes()})
    for cid in sorted(set(lab.values())):
        members = [n for n, c in lab.items() if c == cid]
        size = len(members)
        cat_counts = Counter([G.nodes[n].get(cat_attr, 'Unknown') for n in members])
        row = {'community': cid, 'size': size}
        for cat in categories:
            row[f'count_{cat}'] = cat_counts.get(cat, 0)
        row['dominant_cat'] = max(cat_counts, key=cat_counts.get) if cat_counts else '--'
        row['dominant_fraction'] = (max(cat_counts.values())/size) if size>0 else np.nan
        rows.append(row)
    df = pd.DataFrame(rows).sort_values('size', ascending=False)
    return df, categories

comp_h_df, _ = _community_category_table(G_hu, lab_h)
comp_tw_df, _ = _community_category_table(G_twu, lab_tw)

print("Hyperlink community category composition (top 5 rows):")
display(comp_h_df.head())
print("Twitter community category composition (top 5 rows):")
display(comp_tw_df.head())

# --------------------------------------------------
# 4. Export metrics (including stability + modularity) to DataFrame
# --------------------------------------------------
assert 'metrics' in globals(), "Metrics dict from previous cell not found."
export_metrics = metrics.copy()
export_metrics.update({
    'Modularity_H': Q_h,
    'Modularity_TW': Q_tw,
    'Stability_ARI_mean_H': stability_stats['avg_ari_h'],
    'Stability_ARI_std_H': stability_stats['std_ari_h'],
    'Stability_ARI_mean_TW': stability_stats['avg_ari_tw'],
    'Stability_ARI_std_TW': stability_stats['std_ari_tw'],
})
metrics_df = pd.DataFrame([export_metrics]).T.rename(columns={0: 'value'})
print("All metrics:")
display(metrics_df)

In [ ]:
# Sankey diagram mapping Hyperlink communities to Twitter communities
import plotly.graph_objects as go
from IPython.display import display, HTML
from collections import Counter
import matplotlib.colors as mcolors

assert 'lab_h' in globals() and 'lab_tw' in globals(), "Run the Directed Louvain cell first."
if 'common_nodes' in globals():
    common_pool = sorted(common_nodes)
else:
    common_pool = sorted(set(lab_h.keys()).intersection(lab_tw.keys()))

flows = Counter()
for node in common_pool:
    comm_h = lab_h.get(node)
    comm_tw = lab_tw.get(node)
    if comm_h is None or comm_tw is None:
        continue
    flows[(comm_h, comm_tw)] += 1
if not flows:
    raise ValueError("No overlapping nodes with community labels to plot.")

hyper_comms = sorted({h for h, _ in flows})
twitter_comms = sorted({t for _, t in flows})
count_h = {c: 0 for c in hyper_comms}
count_tw = {c: 0 for c in twitter_comms}
for (comm_h, comm_tw), value in flows.items():
    count_h[comm_h] += value
    count_tw[comm_tw] += value

_palette = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
    '#393b79','#637939','#8c6d31','#843c39','#7b4173','#3182bd','#e6550d','#31a354','#756bb1','#636363'
]

def _ensure_color_map(comm_list, existing):
    cmap = {}
    for idx, comm in enumerate(comm_list):
        if isinstance(existing, dict) and comm in existing:
            cmap[comm] = existing[comm]
        else:
            cmap[comm] = _palette[idx % len(_palette)]
    return cmap

cmap_h_local = _ensure_color_map(hyper_comms, globals().get('cmap_h'))
cmap_tw_local = _ensure_color_map(twitter_comms, globals().get('cmap_tw'))

node_labels = [f"Hyperlink c{cid} (n={count_h[cid]})" for cid in hyper_comms]
node_labels += [f"Twitter c{cid} (n={count_tw[cid]})" for cid in twitter_comms]
node_colors = [cmap_h_local[cid] for cid in hyper_comms] + [cmap_tw_local[cid] for cid in twitter_comms]

hyper_index = {cid: idx for idx, cid in enumerate(hyper_comms)}
twitter_index = {cid: len(hyper_comms) + idx for idx, cid in enumerate(twitter_comms)}

links_sorted = sorted(flows.items(), key=lambda kv: (kv[0][0], kv[0][1]))
sources, targets, values, link_labels, link_colors = [], [], [], [], []
for (comm_h, comm_tw), value in links_sorted:
    sources.append(hyper_index[comm_h])
    targets.append(twitter_index[comm_tw])
    values.append(value)
    link_labels.append(f"{value} nodes: H{comm_h} -> T{comm_tw}")
    rgba = mcolors.to_rgba(cmap_h_local.get(comm_h, '#999999'), alpha=0.55)
    link_colors.append(f"rgba({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)}, {rgba[3]:.2f})")

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=node_labels,
        color=node_colors,
        pad=22,
        thickness=18
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        label=link_labels,
        color=link_colors
    )
))
fig.update_layout(
    title="Community membership flow: Hyperlink -> Twitter",
    font=dict(size=12)
 )
display(HTML(fig.to_html(include_plotlyjs='cdn')))
print(f"Sankey includes {len(values)} flows across {len(hyper_comms)} Hyperlink and {len(twitter_comms)} Twitter communities.")

In [ ]:
# Winner graphs colored by categories (shared layout)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx

assert 'pos_shared' in globals(), "Compute the community layout first."
assert 'color_map_cat' in globals(), "Category color map not defined."

cat_default_color = '#bdbdbd'
edge_kwargs = dict(width=0.4, alpha=0.25, arrows=False)

fig_cat_h, ax_cat_h = plt.subplots(figsize=(12, 12))
ax_cat_h.set_title('Hyperlink winners – categories')
nx.draw_networkx_edges(G_h_dir, pos_shared, ax=ax_cat_h, **edge_kwargs)
colors_h = [color_map_cat.get(G_h_dir.nodes[n].get('category'), cat_default_color) for n in G_h_dir.nodes()]
node_sizes_h, styled_colors_h, seed_labels_h = _style_nodes(G_h_dir, colors_h)
nx.draw_networkx_nodes(G_h_dir, pos_shared, ax=ax_cat_h,
                       node_color=styled_colors_h,
                       node_size=node_sizes_h,
                       edgecolors='black', linewidths=0.6)
if seed_labels_h:
    nx.draw_networkx_labels(G_h_dir, pos_shared, labels=seed_labels_h,
                            font_size=10, font_color='black', ax=ax_cat_h)
ax_cat_h.axis('off')
handles_h = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                        markerfacecolor=color_map_cat.get(cat, cat_default_color),
                        markeredgecolor='black', label=cat)
             for cat in sorted({G_h_dir.nodes[n].get('category', 'Unknown') for n in G_h_dir.nodes()})]
if handles_h:
    ax_cat_h.legend(handles=handles_h, frameon=False, loc='upper left', ncol=2, title='Category')
plt.show()

fig_cat_tw, ax_cat_tw = plt.subplots(figsize=(12, 12))
ax_cat_tw.set_title('Twitter winners – categories')
nx.draw_networkx_edges(G_tw_dir, pos_shared, ax=ax_cat_tw, **edge_kwargs)
colors_tw = [color_map_cat.get(G_tw_dir.nodes[n].get('category'), cat_default_color) for n in G_tw_dir.nodes()]
node_sizes_tw, styled_colors_tw, seed_labels_tw = _style_nodes(G_tw_dir, colors_tw)
nx.draw_networkx_nodes(G_tw_dir, pos_shared, ax=ax_cat_tw,
                       node_color=styled_colors_tw,
                       node_size=node_sizes_tw,
                       edgecolors='black', linewidths=0.6)
if seed_labels_tw:
    nx.draw_networkx_labels(G_tw_dir, pos_shared, labels=seed_labels_tw,
                            font_size=10, font_color='black', ax=ax_cat_tw)
ax_cat_tw.axis('off')
handles_tw = [plt.Line2D([0],[0], marker='o', linestyle='', markersize=10,
                        markerfacecolor=color_map_cat.get(cat, cat_default_color),
                        markeredgecolor='black', label=cat)
             for cat in sorted({G_tw_dir.nodes[n].get('category', 'Unknown') for n in G_tw_dir.nodes()})]
if handles_tw:
    ax_cat_tw.legend(handles=handles_tw, frameon=False, loc='upper left', ncol=2, title='Category')
plt.show()

print("Category-colored visualizations generated (shared layout reused).")